# `廣論逐字稿.ipynb` 使用說明

這個 notebook 用來處理中藏雙語音檔：先把長音檔切成固定長度的小檔，再做語音活動偵測（VAD）、語言判斷（LID），最後分別用中文與藏文模型輸出逐字稿與字幕。

## Notebook 流程
1. 指定原始 MP3 檔名並切成多個片段。
2. 針對每個 5 分鐘切片依序做 VAD，找出實際有人聲的時間區段。
3. 用中文模型的底層語言偵測，把每段標成 `zh` 或 `bo`。
4. 依語言合併片段，交給對應的 ASR 與對齊模型處理。
5. 把所有切片結果整合回原始音檔的全域時間軸，輸出 CSV、JSON 與 SRT。

> notebook 目前會依序處理 `audio_cuts` 裡的所有切片，最後輸出的時間戳會自動加回原始音檔的全域時間。


## 0. 環境建置

第一次執行前，建議先把 Python 套件、`gcloud` 認證與 Vertex 相關環境變數準備好。

可直接在 notebook 跑安裝指令：

```python
!pip install ffmpeg-python whisperx pyannote.audio google-genai google-cloud-storage opencc-python-reimplemented pyewts
```

如果是用 terminal，比較建議在目前的 conda / venv 環境執行：

```bash
pip install ffmpeg-python whisperx pyannote.audio google-genai google-cloud-storage opencc-python-reimplemented pyewts
```

若啟用 Vertex AI 藏語轉寫，還需要先完成 Google Cloud 認證：

```bash
gcloud auth application-default login
gcloud auth application-default set-quota-project YOUR_GCP_PROJECT_ID
```

常用環境變數範例：

```bash
export GOOGLE_CLOUD_PROJECT="YOUR_GCP_PROJECT_ID"
export VERTEX_LOCATION="global"
export VERTEX_STAGING_GCS_PREFIX="gs://YOUR_BUCKET/vertex-staging"
# 可選：若要走 express mode 再設這個
# export GOOGLE_CLOUD_API_KEY="YOUR_GOOGLE_CLOUD_API_KEY"
```

裝完套件或改完環境變數後，記得重開 kernel 再重新執行 notebook。


## 1. 全域開關與準備輸入音檔

- `mp3_audio`：原始輸入音檔名稱。
- `TRANSCRIBE_ZH_INTERPRETER`：**是否聽寫現場中文譯師（口譯員）的部分**（放在最開頭方便直接切換）：
  - `True`：同時聽寫並輸出「格西藏文＋藏翻中直譯」與「現場中文譯師口譯」。
  - `False`：**不聽寫中文譯師**（完全跳過中文 WhisperX 聽寫與對齊模型，只輸出格西藏文＋藏翻中直譯）。
- `trim_duration`：目前設成 `60 * 5`，也就是每 5 分鐘切一段並寫入 `audio_5min/`。

In [1]:
# 1. 指定原始音檔名稱
mp3_audio = "900121c_P001.mp3"

# 2. 是否聽寫現場中文譯師（口譯員）的部分？
#    - True ：保留並聽寫現場中文譯師（同時輸出「格西藏文＋藏翻中直譯」與「現場中文譯師口譯」）
#    - False：不需要聽寫中文譯師（跳過中文 ASR 與對齊，只輸出「格西藏文＋藏翻中直譯」）
TRANSCRIBE_ZH_INTERPRETER = False

In [2]:
import ffmpeg
import math
import os

In [3]:
trim_duration_min = 5
trim_duration = 60 * trim_duration_min
output_dir = f"audio_{trim_duration_min}min"
os.makedirs(output_dir, exist_ok=True)

# 清理舊的切片檔案，避免不同長度音檔殘留（如 12_3600_3900.mp3）混入後續流程
for stale_file in os.listdir(output_dir):
    if stale_file.endswith(".mp3"):
        try:
            os.remove(os.path.join(output_dir, stale_file))
        except OSError:
            pass

probe = ffmpeg.probe(mp3_audio)
stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'audio'), None)
duration = math.floor(float(stream['duration']))
num_cuts = math.ceil(duration / trim_duration)
print(f"Total duration: {duration} sec")
print(f"Trim into {num_cuts} files by {trim_duration_min} min ({trim_duration} sec) each")

audio_input = ffmpeg.input(mp3_audio)
flag = 0
generated_cuts = []
for cut in range(num_cuts):
    if flag + trim_duration > duration:
        end = duration
    else:
        end = flag + trim_duration
    out_file = f'{output_dir}/{cut}_{flag}_{end}.mp3'
    audio_cut = audio_input.audio.filter('atrim', start=flag, end=end)
    audio_output = ffmpeg.output(audio_cut, out_file)
    ffmpeg.run(audio_output, overwrite_output=True)
    generated_cuts.append(out_file)
    flag += trim_duration

Total duration: 3779 sec
Trim into 13 files by 5 min (300 sec) each


ffmpeg version 5.1.6 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix='/usr/local/Cellar/ffmpeg@5/5.1.6_7' --datadir='/usr/local/Cellar/ffmpeg@5/5.1.6_7/share/ffmpeg' --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-

In [4]:
import glob
audio_cuts = glob.glob(f"{output_dir}/*.mp3")
audio_cuts = sorted(
    audio_cuts,
    key=lambda x: tuple(int(v) for v in os.path.splitext(os.path.basename(x))[0].split("_")[:3]),
)
print(audio_cuts)
print(f"find totally audio_cut: {len(audio_cuts)}")


['audio_5min/0_0_300.mp3', 'audio_5min/1_300_600.mp3', 'audio_5min/2_600_900.mp3', 'audio_5min/3_900_1200.mp3', 'audio_5min/4_1200_1500.mp3', 'audio_5min/5_1500_1800.mp3', 'audio_5min/6_1800_2100.mp3', 'audio_5min/7_2100_2400.mp3', 'audio_5min/8_2400_2700.mp3', 'audio_5min/9_2700_3000.mp3', 'audio_5min/10_3000_3300.mp3', 'audio_5min/11_3300_3600.mp3', 'audio_5min/12_3600_3779.mp3', 'audio_5min/12_3600_3900.mp3', 'audio_5min/13_3900_3903.mp3']
find totally audio_cut: 15


In [5]:
# 若只想快速測試前 3 段，可取消下一行註解
audio_cuts = audio_cuts
audio_cuts

['audio_5min/0_0_300.mp3',
 'audio_5min/1_300_600.mp3',
 'audio_5min/2_600_900.mp3',
 'audio_5min/3_900_1200.mp3',
 'audio_5min/4_1200_1500.mp3',
 'audio_5min/5_1500_1800.mp3',
 'audio_5min/6_1800_2100.mp3',
 'audio_5min/7_2100_2400.mp3',
 'audio_5min/8_2400_2700.mp3',
 'audio_5min/9_2700_3000.mp3',
 'audio_5min/10_3000_3300.mp3',
 'audio_5min/11_3300_3600.mp3',
 'audio_5min/12_3600_3779.mp3',
 'audio_5min/12_3600_3900.mp3',
 'audio_5min/13_3900_3903.mp3']

## 2. 核心工具函式

這個區塊把整個流程需要的 helper function 先定義好，後面的 cell 只要專注在串流程。

重點函式如下：
- `run_vad_segments`：用 VAD 找出有人聲的區段。
- `detect_zh_bo`：用滑動視窗累積語言分數，做 `zh` / `bo` 二分類。
- `build_lid_blocks`、`smooth_langs_binary`：把零碎語音段整理成較穩定的語言標記。
- `merge_for_asr`：把相鄰且同語言的區段合併，降低 ASR 呼叫次數。
- `transcribe_block_fast`、`transcribe_and_align`：轉寫與時間對齊。
- `parse_audio_cut_path`、`shift_time_rows`、`run_pipeline_for_audio_cut`：把每個切片結果換算回全域時間並整合。
- `write_csv`、`write_srt`：把結果輸出成後處理容易使用的格式。


In [6]:
def normalize_lang(lang: str) -> str:
    lang = (lang or "").lower()
    if lang.startswith("zh"):
        return "zh"
    if lang.startswith("bo"):
        return "bo"
    return "other"


def detect_language_no_warning(pipeline, audio_chunk):
    model_n_mels = pipeline.model.feat_kwargs.get("feature_size")
    segment = log_mel_spectrogram(
        audio_chunk[:N_SAMPLES],
        n_mels=model_n_mels if model_n_mels is not None else 80,
        padding=0 if audio_chunk.shape[0] >= N_SAMPLES else N_SAMPLES - audio_chunk.shape[0],
    )
    encoder_output = pipeline.model.encode(segment)
    results = pipeline.model.model.detect_language(encoder_output)
    language_token, probability = results[0][0]
    return language_token[2:-2], float(probability)


def run_vad_segments(audio, vad_model, chunk_size=8, vad_onset=0.45, vad_offset=0.30):
    """細粒度 VAD：預設將 chunk_size 由 30s 降至 8s，避免把中藏語切換黏在同一個 30s 區塊內。"""
    if issubclass(type(vad_model), Vad):
        waveform = vad_model.preprocess_audio(audio)
        merge_chunks = vad_model.merge_chunks
    else:
        waveform = Pyannote.preprocess_audio(audio)
        merge_chunks = Pyannote.merge_chunks

    vad_scores = vad_model({"waveform": waveform, "sample_rate": SAMPLE_RATE})
    merged = merge_chunks(vad_scores, chunk_size, onset=vad_onset, offset=vad_offset)

    segs = []
    for s in merged:
        start = float(s["start"])
        end = float(s["end"])
        if end - start >= 0.2:
            segs.append({"start": start, "end": end})
    return segs


def _lang_probs(pipeline, chunk):
    n_mels = pipeline.model.feat_kwargs.get("feature_size") or 80
    mel = log_mel_spectrogram(
        chunk[:N_SAMPLES],
        n_mels=n_mels,
        padding=max(0, N_SAMPLES - len(chunk)),
    )
    enc = pipeline.model.encode(mel)
    # 回傳格式: [("zh", 0.7), ("en", 0.2), ...]
    return [(tok[2:-2], float(p)) for tok, p in pipeline.model.model.detect_language(enc)[0]]


def detect_zh_bo(
    pipeline,
    chunk,
    win_sec=4.5,
    hop_sec=4.0,
    min_win_sec=0.4,
    zh_low_th=0.97,      # 真實中文口譯在 large-v3-turbo 下 zh_r 幾乎皆 >= 0.995；低於 0.97 多為藏文被誤判
    other_min_th=0.03,   # 只要有非中文機率或藏文機率即優先歸為 bo（高召回保護藏文）
):
    votes = {"zh": 0.0, "bo": 0.0, "other": 0.0}
    win = int(win_sec * SAMPLE_RATE)
    hop = int(hop_sec * SAMPLE_RATE)
    min_len = int(min_win_sec * SAMPLE_RATE)

    if len(chunk) <= int(6.5 * SAMPLE_RATE):
        ranges = [(0, len(chunk))]
    else:
        ranges = [(i, min(i + win, len(chunk))) for i in range(0, len(chunk) - win + 1, hop)]
        if ranges and ranges[-1][1] < len(chunk) and (len(chunk) - ranges[-1][0]) >= min_len:
            ranges.append((max(0, len(chunk) - win), len(chunk)))

    for a, b in ranges:
        sub = chunk[a:b]
        if len(sub) < min_len:
            continue
        for lang, prob in _lang_probs(pipeline, sub):
            if lang.startswith("zh"):
                votes["zh"] += prob
            elif lang.startswith("bo"):
                votes["bo"] += prob
            else:
                votes["other"] += prob

    total = votes["zh"] + votes["bo"] + votes["other"]
    if total == 0:
        return "other", 0.0, votes

    zh_r = votes["zh"] / total
    bo_r = votes["bo"] / total
    other_r = votes["other"] / total

    # 高召回藏語規則：只要 bo 顯著，或 zh 未達高信心門檻 (0.97) 且帶有非漢語特徵 => bo
    if bo_r >= 0.02 or (zh_r < zh_low_th and (other_r + bo_r) >= other_min_th):
        return "bo", max(bo_r, 1.0 - zh_r), votes

    if bo_r >= zh_r:
        return "bo", bo_r, votes
    return "zh", zh_r, votes


def split_vad_segments_on_lang_change(
    lid_pipeline,
    audio,
    vad_segments,
    min_split_sec=6.0,
    sub_win_sec=3.0,
    sub_hop_sec=1.5,
):
    """對較長的 VAD 區段進行子視窗掃描；若區段內部發生中藏語切換，於切換點自動切開。"""
    refined = []
    for seg in vad_segments:
        st, ed = float(seg["start"]), float(seg["end"])
        dur = ed - st
        if dur < min_split_sec:
            refined.append({"start": st, "end": ed})
            continue

        # 先快速檢查前半段與後半段是否同語言；若同語言則無需逐 1.5s 細掃，速度提升 8 倍
        mid = 0.5 * (st + ed)
        l_first, _, _ = detect_zh_bo(lid_pipeline, audio[int(st * SAMPLE_RATE):int(mid * SAMPLE_RATE)])
        l_second, _, _ = detect_zh_bo(lid_pipeline, audio[int(mid * SAMPLE_RATE):int(ed * SAMPLE_RATE)])
        if l_first == l_second and l_first in {"zh", "bo"}:
            refined.append({"start": st, "end": ed})
            continue

        sub_points = []
        t = st
        while t + 1.0 < ed:
            w_end = min(ed, t + sub_win_sec)
            if w_end - t < 1.0 and sub_points:
                break
            s1, s2 = int(t * SAMPLE_RATE), int(w_end * SAMPLE_RATE)
            lang, conf, _ = detect_zh_bo(lid_pipeline, audio[s1:s2], win_sec=sub_win_sec, hop_sec=sub_win_sec)
            sub_points.append((t, w_end, lang, conf))
            if w_end >= ed:
                break
            t += sub_hop_sec

        langs_in_seg = {p[2] for p in sub_points if p[2] in {"zh", "bo"}}
        if len(langs_in_seg) <= 1 or len(sub_points) < 2:
            refined.append({"start": st, "end": ed})
            continue

        # 抹平單一孤立跳動點
        labels = [p[2] for p in sub_points]
        for idx in range(1, len(labels) - 1):
            if labels[idx - 1] == labels[idx + 1] and labels[idx] != labels[idx - 1]:
                labels[idx] = labels[idx - 1]

        cur_start = st
        cur_lang = labels[0]
        for idx in range(1, len(labels)):
            if labels[idx] != cur_lang:
                split_time = round(sub_points[idx][0], 3)
                if split_time - cur_start >= 1.2 and ed - split_time >= 1.2:
                    refined.append({"start": cur_start, "end": split_time})
                    cur_start = split_time
                    cur_lang = labels[idx]
        refined.append({"start": cur_start, "end": ed})

    return refined


def build_lid_blocks(vad_segments, min_lid_sec=1.8, max_gap=0.35, max_block_sec=8.0):
    blocks = []
    i, n = 0, len(vad_segments)
    while i < n:
        st = vad_segments[i]["start"]
        ed = vad_segments[i]["end"]
        idxs = [i]
        j = i + 1
        while j < n:
            gap = vad_segments[j]["start"] - ed
            new_ed = vad_segments[j]["end"]
            if gap <= max_gap and (new_ed - st) <= max_block_sec and (ed - st) < min_lid_sec:
                ed = new_ed
                idxs.append(j)
                j += 1
            else:
                break
        blocks.append({"start": st, "end": ed, "idxs": idxs})
        i = j
    return blocks


def detect_block_language_zh_bo(lid_pipeline, audio, start, end):
    s1, s2 = int(start * SAMPLE_RATE), int(end * SAMPLE_RATE)
    chunk = audio[s1:s2]
    if len(chunk) < int(0.3 * SAMPLE_RATE):
        return "other", 0.0, {"zh": 0.0, "bo": 0.0}
    return detect_zh_bo(lid_pipeline, chunk)


def smooth_langs_binary(
    seg_lang,
    seg_conf=None,
    vad_segments=None,
    default_lang="bo",
    max_zh_island_sec=8.0,
    zh_strict_conf=0.992,
):
    """平滑語言標籤：除了填補 other，也會把夾在藏語旁、信心度不足 (<0.992) 的短暫偽中文段校正回 bo。"""
    out = seg_lang[:]
    n = len(out)
    if n == 0:
        return out

    for i in range(n):
        if out[i] in {"zh", "bo"}:
            continue
        left = next((out[j] for j in range(i - 1, -1, -1) if out[j] in {"zh", "bo"}), None)
        right = next((out[j] for j in range(i + 1, n) if out[j] in {"zh", "bo"}), None)
        if left == right and left is not None:
            out[i] = left
        elif left == "bo" or right == "bo":
            out[i] = "bo"

    for i in range(n):
        if out[i] not in {"zh", "bo"}:
            out[i] = out[i - 1] if i > 0 else default_lang

    # 二次校正：若短暫 zh 段落信心度 < zh_strict_conf (0.992) 且緊鄰 bo，多為藏語尾句被 Whisper 幻覺成中文
    if seg_conf is not None and vad_segments is not None:
        for i in range(n):
            dur = float(vad_segments[i]["end"]) - float(vad_segments[i]["start"])
            conf = float(seg_conf[i])
            if out[i] == "zh" and dur <= max_zh_island_sec and conf < zh_strict_conf:
                left_bo = (i > 0 and out[i - 1] == "bo")
                right_bo = (i + 1 < n and out[i + 1] == "bo")
                if left_bo or right_bo:
                    out[i] = "bo"
            elif out[i] == "bo" and dur <= 2.5 and conf < 0.60 and 0 < i < n - 1:
                if out[i - 1] == "zh" and out[i + 1] == "zh":
                    if float(seg_conf[i - 1]) >= 0.995 and float(seg_conf[i + 1]) >= 0.995:
                        out[i] = "zh"

    return out


def merge_for_asr(
    vad_segments,
    seg_lang,
    merge_gap=0.6,
    min_asr_sec=0.8,
    max_block_sec=40.0,
    bo_merge_gap=2.5,
    bo_min_asr_sec=0.25,
    bo_max_block_sec=35.0,
):
    """依語言分開設定合併策略：
    - 藏語 (bo) 允許跨 2.5 秒短暫呼吸停頓合併成 25~35 秒的純藏語區塊，最利於 Gemini 3.1 Pro 穩定聽寫且不截斷。
    - 漢語 (zh) 維持較緊湊的合併送給本地 WhisperX + Align。
    """
    rows = []
    for s, lang in zip(vad_segments, seg_lang):
        if lang not in {"zh", "bo"}:
            continue
        rows.append({"start": float(s["start"]), "end": float(s["end"]), "lang": lang})

    if not rows:
        return []

    merged = [rows[0]]
    for cur in rows[1:]:
        prev = merged[-1]
        same_lang = cur["lang"] == prev["lang"]
        gap = cur["start"] - prev["end"]
        new_len = cur["end"] - prev["start"]
        cur_max_gap = bo_merge_gap if cur["lang"] == "bo" else merge_gap
        cur_max_len = bo_max_block_sec if cur["lang"] == "bo" else max_block_sec
        if same_lang and gap <= cur_max_gap and new_len <= cur_max_len:
            prev["end"] = cur["end"]
        else:
            merged.append(cur)

    final_blocks = []
    for m in merged:
        min_len = bo_min_asr_sec if m["lang"] == "bo" else min_asr_sec
        if (m["end"] - m["start"]) >= min_len:
            final_blocks.append(m)
    return final_blocks


def pad_bo_blocks(bo_blocks, all_blocks, audio_duration_sec, pad_start_sec=0.5, pad_end_sec=0.6):
    """為每個藏語區塊前後加上安全緩衝 (0.5s~0.6s)，但不侵入相鄰的漢語區塊，防止格西起音與結尾語助詞被切掉。"""
    if not bo_blocks:
        return []
    zh_blocks = sorted([b for b in all_blocks if b["lang"] == "zh"], key=lambda x: x["start"])
    padded = []
    for idx, b in enumerate(bo_blocks):
        st = float(b["start"])
        ed = float(b["end"])
        min_st = 0.0
        max_ed = float(audio_duration_sec)
        for zb in zh_blocks:
            if float(zb["end"]) <= st:
                min_st = max(min_st, float(zb["end"]))
            if float(zb["start"]) >= ed:
                max_ed = min(max_ed, float(zb["start"]))
                break
        if idx > 0:
            min_st = max(min_st, float(padded[-1]["end"]))
        if idx + 1 < len(bo_blocks):
            max_ed = min(max_ed, float(bo_blocks[idx + 1]["start"]))

        new_st = max(min_st, st - pad_start_sec)
        new_ed = min(max_ed, ed + pad_end_sec)
        item = b.copy()
        item["start"] = round(new_st, 3)
        item["end"] = round(new_ed, 3)
        padded.append(item)
    return padded


def clean_wylie_repetitions(text: str) -> str:
    """清除藏文 Wylie 聲學模型在句尾可能產生的重複疊字（例如 red red red 或詞組循環）。"""
    text = (text or "").strip()
    if not text:
        return ""
    words = text.split()
    if len(words) <= 2:
        return text
    changed = True
    while changed:
        changed = False
        for n in range(min(8, len(words) // 2), 0, -1):
            i = 0
            out = []
            while i < len(words):
                gram = words[i : i + n]
                if i + 2 * n <= len(words) and words[i + n : i + 2 * n] == gram:
                    out.extend(gram)
                    i += n
                    while i + n <= len(words) and words[i : i + n] == gram:
                        i += n
                    changed = True
                else:
                    out.append(words[i])
                    i += 1
            words = out
    return " ".join(words)


class MPSWhisperPipeline:
    """在 Apple Silicon (M1/M2/M3/M4) MPS GPU 上執行藏語 Whisper 批次平行推論的封裝類別。
    使用 GenerationMixin.generate 搭配明確的藏語 decoder_input_ids，避開 WhisperGenerationMixin 的時間戳尋軌死鎖，
    並以 float32 避免 MPS SDPA Cross-Attention 在 16-bit 下的數值溢位問題。
    """
    def __init__(self, model_id="openpecha/op-whisper_small-ft-v2", device="mps", batch_size=6):
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        from transformers.generation import GenerationMixin
        self.model_id = model_id
        self.device = device
        self.batch_size = batch_size
        self._gen_fn = GenerationMixin.generate
        self.processor = WhisperProcessor.from_pretrained(model_id)
        self.model = WhisperForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
        ).to(device).eval()
        self.model.generation_config.forced_decoder_ids = None
        self.model.config.forced_decoder_ids = None
        # <|startoftranscript|>=50258, <|bo|>=50347, <|transcribe|>=50359, <|notimestamps|>=50363
        self.prompt_ids = [50258, 50347, 50359, 50363]

    def transcribe_batch(self, audio_chunks):
        if not audio_chunks:
            return []
        results = []
        for i in range(0, len(audio_chunks), self.batch_size):
            batch = audio_chunks[i : i + self.batch_size]
            inputs = self.processor(batch, sampling_rate=SAMPLE_RATE, return_tensors="pt")
            feats = inputs.input_features.to(self.device, dtype=torch.float32)
            dec_ids = torch.tensor([self.prompt_ids] * len(batch), dtype=torch.long, device=self.device)
            with torch.no_grad():
                gen_ids = self._gen_fn(
                    self.model,
                    inputs=feats,
                    decoder_input_ids=dec_ids,
                    max_new_tokens=96,
                    do_sample=False,
                    num_beams=1,
                    repetition_penalty=1.35,
                    no_repeat_ngram_size=3,
                    length_penalty=1.0,
                )
            if self.device == "mps":
                torch.mps.synchronize()
            decoded = self.processor.batch_decode(gen_ids, skip_special_tokens=True)
            for txt in decoded:
                t = (txt or "").strip()
                if t and t.isascii():
                    t = clean_wylie_repetitions(t)
                results.append(t)
        return results


def _transcribe_single_slice(asr_pipeline, sub_chunk, lang):
    if isinstance(asr_pipeline, MPSWhisperPipeline):
        txts = asr_pipeline.transcribe_batch([sub_chunk])
        t = txts[0] if txts else ""
        return [{"start": 0.0, "end": round(len(sub_chunk) / float(SAMPLE_RATE), 3), "text": t}] if t else []
    extra_kwargs = {}
    if lang == "bo":
        extra_kwargs = {
            "repetition_penalty": 1.35,
            "no_repeat_ngram_size": 3,
        }
    try:
        seg_iter, _ = asr_pipeline.model.transcribe(
            sub_chunk,
            language=lang,
            beam_size=5,
            vad_filter=False,
            condition_on_previous_text=False,
            **extra_kwargs,
        )
    except TypeError:
        seg_iter, _ = asr_pipeline.model.transcribe(
            sub_chunk,
            language=lang,
            beam_size=5,
            condition_on_previous_text=False,
        )
    out = []
    for s in seg_iter:
        text = (s.text or "").strip()
        if lang == "bo" and text:
            text = clean_wylie_repetitions(text)
        if text:
            out.append({"start": float(s.start), "end": float(s.end), "text": text})
    return out


def split_audio_at_pauses(
    audio_chunk,
    sample_rate=16000,
    target_sec=9.0,
    min_sec=6.0,
    max_sec=11.0,
    tail_merge_sec=1.8,
    window_ms=80,
):
    """智慧呼吸停頓切點（Minimum RMS Energy Pause）：
    在 [min_sec, max_sec] 區間內自動尋找 80ms 音量能量最低的換氣停頓點下刀，
    避免固定 9.0 秒硬切把格西正在發音的藏文音節砍成兩半。
    回傳 list of (start_sample, end_sample)。
    """
    total = len(audio_chunk)
    max_samples = int(max_sec * sample_rate)
    if total <= max_samples:
        return [(0, total)]

    min_samples = int(min_sec * sample_rate)
    target_samples = int(target_sec * sample_rate)
    tail_merge_samples = int(tail_merge_sec * sample_rate)
    win_len = max(160, int(sample_rate * window_ms / 1000.0))
    hop_len = max(80, win_len // 2)

    slices = []
    offset = 0
    while offset < total:
        rem = total - offset
        if rem <= max_samples:
            slices.append((offset, total))
            break

        search_start = offset + min_samples
        search_end = min(total, offset + max_samples)
        region = audio_chunk[search_start:search_end]

        if len(region) >= win_len:
            best_pos = offset + target_samples
            best_score = float("inf")
            for i in range(0, len(region) - win_len + 1, hop_len):
                frame = region[i : i + win_len]
                rms = float((frame * frame).mean() ** 0.5)
                cand_pos = search_start + i + win_len // 2
                dist_penalty = 0.002 * abs(cand_pos - (offset + target_samples)) / float(sample_rate)
                score = rms + dist_penalty
                if score < best_score:
                    best_score = score
                    best_pos = cand_pos
            cut = best_pos
        else:
            cut = min(total, offset + target_samples)

        if total - cut < tail_merge_samples:
            cut = total
        slices.append((offset, cut))
        offset = cut

    return slices


def transcribe_block_fast(asr_pipeline, audio_chunk, lang, bo_slice_sec=9.0):
    """針對藏文聲學模型 (openpecha_op-whisper_small-ft-v2)，使用智慧呼吸停頓切點切為 ~9.0 秒短窗並啟用防重複懲罰。"""
    if lang == "bo" and len(audio_chunk) > int(11.0 * SAMPLE_RATE):
        segments = []
        for s_off, e_off in split_audio_at_pauses(audio_chunk, sample_rate=SAMPLE_RATE, target_sec=bo_slice_sec):
            sub_chunk = audio_chunk[s_off:e_off]
            offset_sec = s_off / float(SAMPLE_RATE)
            sub_segs = _transcribe_single_slice(asr_pipeline, sub_chunk, lang)
            for s in sub_segs:
                segments.append({
                    "start": round(offset_sec + float(s["start"]), 3),
                    "end": round(offset_sec + float(s["end"]), 3),
                    "text": s["text"],
                })
        return segments

    return _transcribe_single_slice(asr_pipeline, audio_chunk, lang)


def audio_chunk_to_wav_bytes(audio_chunk, sample_rate=None):
    if sample_rate is None:
        sample_rate = globals().get("SAMPLE_RATE", 16000)
    pcm16 = (audio_chunk.clip(-1.0, 1.0) * 32767.0).astype("int16")
    with io.BytesIO() as buffer:
        with wave.open(buffer, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(pcm16.tobytes())
        return buffer.getvalue()


def build_transcript_row(start, end, lang, text="", text_bo="", text_zh="", source=""):
    text = (text or "").strip()
    text_bo = (text_bo or "").strip()
    text_zh = (text_zh or "").strip()
    if lang == "zh" and not text_zh:
        text_zh = text
    if lang == "bo" and not text:
        text = text_zh or text_bo
    return {
        "start": round(float(start), 3),
        "end": round(float(end), 3),
        "lang": lang,
        "source": source,
        "text": text,
        "text_bo": text_bo,
        "text_zh": text_zh,
    }


def split_block_evenly(start, end, count):
    if count <= 0:
        return []
    if count == 1:
        return [(round(float(start), 3), round(float(end), 3))]
    total = max(float(end) - float(start), 0.0)
    step = total / count if count else total
    spans = []
    for idx in range(count):
        seg_start = float(start) + step * idx
        seg_end = float(end) if idx == count - 1 else float(start) + step * (idx + 1)
        spans.append((round(seg_start, 3), round(seg_end, 3)))
    return spans


def detect_audio_mime_type(audio_path):
    suffix = Path(audio_path).suffix.lower()
    if suffix == ".mp3":
        return "audio/mpeg"
    if suffix == ".wav":
        return "audio/wav"
    if suffix == ".m4a":
        return "audio/mp4"
    return "application/octet-stream"


def ensure_cut_audio_staged(audio_path, vertex_cfg):
    object_name = make_staging_object_name(
        source_name=os.path.basename(audio_path),
        suffix="full_cut",
        extension=Path(audio_path).suffix.lstrip("."),
    )
    gcs_uri = build_gcs_uri(vertex_cfg["staging_gcs_prefix"], object_name)
    upload_file_to_gcs(
        local_path=audio_path,
        gcs_uri=gcs_uri,
        content_type=detect_audio_mime_type(audio_path),
    )
    return gcs_uri


def segment_overlaps_blocks(start_sec, end_sec, blocks):
    for block in blocks:
        overlap = max(0.0, min(end_sec, float(block["end"])) - max(start_sec, float(block["start"])))
        if overlap > 0.0:
            return True
    return False


def total_duration(rows_or_blocks):
    total = 0.0
    for item in rows_or_blocks:
        total += max(0.0, float(item["end"]) - float(item["start"]))
    return total


def _format_rolling_context(rows, max_items=4):
    lines = []
    for r in rows[-max_items:]:
        bo_txt = (r.get("text_bo") or r.get("tibetan_text") or "").strip()
        zh_txt = (r.get("text_zh") or r.get("translation_zh") or "").strip()
        if bo_txt:
            lines.append(f"- 藏文：{bo_txt}" + (f" ｜ 中譯：{zh_txt}" if zh_txt else ""))
    return "\n".join(lines)


def transcribe_bo_with_vertex_mixed_cut(audio_path, bo_blocks, vertex_cfg):
    if not bo_blocks:
        return []

    staged_uri = ensure_cut_audio_staged(audio_path, vertex_cfg)
    two_pass = bool(vertex_cfg.get("two_pass", True))
    reference_text = str(vertex_cfg.get("reference_text", "") or "")
    previous_context = str(vertex_cfg.get("_rolling_context", "") or "")
    try:
        segments = transcribe_tibetan_audio_from_gcs(
            gcs_uri=staged_uri,
            project=vertex_cfg["project"],
            location=vertex_cfg["location"],
            api_key=vertex_cfg.get("api_key", ""),
            mime_type=detect_audio_mime_type(audio_path),
            model_name=vertex_cfg["model_name"],
            thinking_level=vertex_cfg.get("thinking_level", "HIGH"),
            max_output_tokens=vertex_cfg.get("max_output_tokens", 65535),
            retry_attempts=vertex_cfg.get("retry_attempts", 5),
            retry_initial_delay_sec=vertex_cfg.get("retry_initial_delay_sec", 5.0),
            retry_backoff_multiplier=vertex_cfg.get("retry_backoff_multiplier", 2.0),
            retry_max_delay_sec=vertex_cfg.get("retry_max_delay_sec", 60.0),
            min_request_interval_sec=vertex_cfg.get("min_request_interval_sec", 2.0),
            attempt_mixed_audio=True,
            two_pass=two_pass,
            reference_text=reference_text,
            previous_context=previous_context,
        )
    finally:
        if vertex_cfg.get("cleanup_staged_objects"):
            try:
                delete_gcs_object(staged_uri)
            except Exception:
                pass

    rows = []
    for seg in segments:
        start_sec = round(timestamp_to_seconds(seg["start_time"]), 3)
        end_sec = round(timestamp_to_seconds(seg["end_time"]), 3)
        text_bo = (seg.get("tibetan_text") or seg.get("text") or "").strip()
        text_zh = (seg.get("translation_zh") or "").strip()
        if not text_bo or end_sec <= start_sec:
            continue
        if not segment_overlaps_blocks(start_sec, end_sec, bo_blocks):
            continue
        rows.append(build_transcript_row(
            start_sec,
            end_sec,
            "bo",
            text=text_zh or text_bo,
            text_bo=text_bo,
            text_zh=text_zh,
            source="vertexai_mixed",
        ))

    rows.sort(key=lambda x: (x["start"], x["end"]))
    if rows:
        vertex_cfg["_rolling_context"] = _format_rolling_context(rows)
    return rows


def transcribe_bo_blocks_with_vertex(audio, audio_path, bo_blocks, vertex_cfg):
    """逐個純藏語區塊送 Vertex AI：
    - 若啟用 two_pass=True（預設）：
      Pass 1 先對每個 25~35s 藏語音訊區塊專注聽寫純藏文（並把前一段藏文作為 previous_context 傳遞給下一段），
      Pass 2 再把整個切片的所有藏文段落整批送交 Gemini 結合《廣論》詞彙表與前後文進行藏文校對與繁體中文精譯。
    """
    two_pass = bool(vertex_cfg.get("two_pass", True))
    reference_text = str(vertex_cfg.get("reference_text", "") or "")
    rolling_context = str(vertex_cfg.get("_rolling_context", "") or "")
    raw_block_segments = []

    for idx, block in enumerate(bo_blocks):
        block_start = float(block["start"])
        block_end = float(block["end"])
        block_dur = block_end - block_start
        s1 = int(block_start * SAMPLE_RATE)
        s2 = int(block_end * SAMPLE_RATE)
        audio_chunk = audio[s1:s2]
        if len(audio_chunk) < int(0.3 * SAMPLE_RATE):
            continue

        object_name = make_staging_object_name(
            source_name=os.path.basename(audio_path),
            suffix=f"bo_{idx}_{int(block_start * 1000)}_{int(block_end * 1000)}",
            extension="wav",
        )
        staged_uri = build_gcs_uri(vertex_cfg["staging_gcs_prefix"], object_name)
        upload_bytes_to_gcs(
            data=audio_chunk_to_wav_bytes(audio_chunk),
            gcs_uri=staged_uri,
            content_type="audio/wav",
        )

        try:
            segments = transcribe_tibetan_audio_from_gcs(
                gcs_uri=staged_uri,
                project=vertex_cfg["project"],
                location=vertex_cfg["location"],
                api_key=vertex_cfg.get("api_key", ""),
                mime_type="audio/wav",
                model_name=vertex_cfg["model_name"],
                thinking_level=vertex_cfg.get("thinking_level", "HIGH"),
                max_output_tokens=vertex_cfg.get("max_output_tokens", 65535),
                retry_attempts=vertex_cfg.get("retry_attempts", 5),
                retry_initial_delay_sec=vertex_cfg.get("retry_initial_delay_sec", 5.0),
                retry_backoff_multiplier=vertex_cfg.get("retry_backoff_multiplier", 2.0),
                retry_max_delay_sec=vertex_cfg.get("retry_max_delay_sec", 60.0),
                min_request_interval_sec=vertex_cfg.get("min_request_interval_sec", 2.0),
                attempt_mixed_audio=False,
                two_pass=False,
                pass1_only=two_pass,
                reference_text=reference_text,
                previous_context=rolling_context,
            )
        finally:
            if vertex_cfg.get("cleanup_staged_objects"):
                try:
                    delete_gcs_object(staged_uri)
                except Exception:
                    pass

        block_valid_segs = []
        for seg in segments:
            rel_st = max(0.0, timestamp_to_seconds(seg["start_time"]))
            rel_ed = min(block_dur + 1.5, timestamp_to_seconds(seg["end_time"]))
            start_sec = round(block_start + rel_st, 3)
            end_sec = round(min(block_end, block_start + rel_ed), 3)
            if end_sec <= start_sec:
                end_sec = round(min(block_end, start_sec + 1.5), 3)
            text_bo = (seg.get("tibetan_text") or seg.get("text") or "").strip()
            text_zh = (seg.get("translation_zh") or "").strip()
            if not text_bo or end_sec <= start_sec:
                continue
            item = {
                "start": start_sec,
                "end": end_sec,
                "start_time": f"{int(start_sec // 60):02d}:{int(start_sec % 60):02d}",
                "end_time": f"{int(end_sec // 60):02d}:{int(end_sec % 60):02d}",
                "tibetan_text": text_bo,
                "text": text_bo,
                "translation_zh": text_zh,
            }
            block_valid_segs.append(item)
            raw_block_segments.append(item)

        if block_valid_segs:
            rolling_context = _format_rolling_context(block_valid_segs)

    if two_pass and raw_block_segments:
        translate_tibetan_segments_to_zh(
            segments=raw_block_segments,
            project=vertex_cfg["project"],
            location=vertex_cfg["location"],
            api_key=vertex_cfg.get("api_key", ""),
            model_name=vertex_cfg["model_name"],
            reference_text=reference_text,
            previous_context=str(vertex_cfg.get("_rolling_context", "") or ""),
            thinking_level=vertex_cfg.get("thinking_level", "HIGH"),
            retry_attempts=vertex_cfg.get("retry_attempts", 5),
            retry_initial_delay_sec=vertex_cfg.get("retry_initial_delay_sec", 5.0),
            retry_backoff_multiplier=vertex_cfg.get("retry_backoff_multiplier", 2.0),
            retry_max_delay_sec=vertex_cfg.get("retry_max_delay_sec", 60.0),
            min_request_interval_sec=vertex_cfg.get("min_request_interval_sec", 2.0),
        )

    rows = []
    for seg in raw_block_segments:
        text_bo = (seg.get("tibetan_text") or seg.get("text") or "").strip()
        text_zh = (seg.get("translation_zh") or "").strip()
        if not text_bo:
            continue
        rows.append(build_transcript_row(
            seg["start"],
            seg["end"],
            "bo",
            text=text_zh or text_bo,
            text_bo=text_bo,
            text_zh=text_zh,
            source="vertexai_two_pass" if two_pass else "vertexai_block",
        ))

    rows.sort(key=lambda x: (x["start"], x["end"]))
    if rows:
        vertex_cfg["_rolling_context"] = _format_rolling_context(rows)
    return rows


_BO_CTC_BEAM_DECODER_CACHE = {}


def get_or_build_bo_ctc_decoder(bo_align_tuple, arpa_path=None):
    """載入並快取 openpecha/op-wav2vec2-ft-v17 專用的 pyctcdecode + BoKenlm-syl-v0.4 音節語言模型解碼器。"""
    if not bo_align_tuple:
        return None
    if arpa_path is None:
        arpa_path = globals().get("BO_KENLM_ARPA_PATH") or str((Path.cwd() / ".cache" / "bokenlm_syl_v04_3gram.arpa").resolve())
    if not arpa_path or not os.path.exists(arpa_path):
        return None
    if arpa_path in _BO_CTC_BEAM_DECODER_CACHE:
        return _BO_CTC_BEAM_DECODER_CACHE[arpa_path]

    try:
        from pyctcdecode import build_ctcdecoder
        from pyctcdecode.language_model import load_unigram_set_from_arpa
        align_model, align_meta = bo_align_tuple
        dictionary = align_meta.get("dictionary", {})
        vocab_size = int(getattr(getattr(align_model, "config", None), "vocab_size", 115) or 115)
        labels = [""] * vocab_size
        for ch, idx in dictionary.items():
            if 0 <= int(idx) < vocab_size:
                labels[int(idx)] = ch
        valid_chars = {c for c in labels if len(c) == 1 and c != " "}
        raw_unigrams = load_unigram_set_from_arpa(arpa_path)
        clean_unigrams = [u for u in raw_unigrams if u and all(c in valid_chars for c in u)]
        decoder = build_ctcdecoder(
            labels=labels,
            kenlm_model_path=str(arpa_path),
            unigrams=clean_unigrams,
            alpha=0.5,
            beta=1.0,
        )
        _BO_CTC_BEAM_DECODER_CACHE[arpa_path] = decoder
        return decoder
    except Exception as exc:
        print(f"[WARN] Failed to initialize BoKenlm CTC Beam Search decoder ({exc}); fallback to greedy CTC.")
        _BO_CTC_BEAM_DECODER_CACHE[arpa_path] = None
        return None


def decode_wav2vec2_ctc(sub_chunk, bo_align_tuple, device="cpu", beam_width=12) -> str:
    """使用 1.28GB 的 openpecha/op-wav2vec2-ft-v17 (Wav2Vec2ForCTC) + BoKenlm-syl-v0.4 音節語言模型進行 CTC Beam Search 解碼。
    優勢：
    1. 逐幀對應音訊，不補齊 30 秒靜音、零句尾幻覺、零無限迴圈。
    2. 掛載 OpenPecha BoKenlm-syl-v0.4（20,564 藏文音節詞典 + N-gram 模型）後，在 Pass 1 即自動將口語連音校正為標準藏文正字（如 སྔོན་ལ་、ཀྱི་、ལུས་ཤིག་）。
    """
    if not bo_align_tuple or len(sub_chunk) < 400:
        return ""
    align_model, align_meta = bo_align_tuple
    dictionary = align_meta.get("dictionary", {})
    if not dictionary:
        return ""
    waveform = torch.from_numpy(sub_chunk).float().unsqueeze(0)
    waveform = (waveform - waveform.mean()) / torch.sqrt(waveform.var() + 1e-7)
    try:
        model_dev = next(align_model.parameters()).device
    except Exception:
        model_dev = device
    waveform = waveform.to(model_dev)
    with torch.no_grad():
        logits_tensor = align_model(waveform).logits[0]

    ctc_decoder = get_or_build_bo_ctc_decoder(bo_align_tuple)
    if ctc_decoder is not None:
        try:
            logits_np = logits_tensor.detach().cpu().numpy()
            beam_raw = ctc_decoder.decode(logits_np, beam_width=beam_width)
            beam_txt = "་".join(w for w in beam_raw.split(" ") if w.strip())
            if beam_txt:
                return beam_txt.strip("་ ") + "་"
        except Exception:
            pass

    id_to_char = {v: k for k, v in dictionary.items()}
    pred_ids = torch.argmax(logits_tensor, dim=-1).tolist()
    collapsed = []
    prev_id = None
    for pid in pred_ids:
        if pid != prev_id:
            ch = id_to_char.get(pid, "")
            if ch and ch.lower() not in {"[pad]", "<pad>", "<s>", "</s>", "[unk]", "<unk>", "|"}:
                collapsed.append("་" if ch == " " else ch)
        prev_id = pid
    text = "".join(collapsed)
    while "་་" in text:
        text = text.replace("་་", "་")
    return text.strip("་ ")


def truncate_by_ctc_syllables(whisper_uni: str, ctc_uni: str, ratio: float = 1.18, extra_syl: int = 2) -> str:
    """利用 CTC 逐幀解碼的真實藏文音節數，截斷 Whisper 因 30 秒補零靜音所產生的句尾幻覺。"""
    if not whisper_uni or not ctc_uni:
        return whisper_uni
    ctc_syls = [s for s in ctc_uni.replace("།", "་").split("་") if s.strip()]
    w_syls = [s for s in whisper_uni.replace("།", "་").split("་") if s.strip()]
    max_syls = max(1, int(len(ctc_syls) * ratio) + extra_syl)
    if len(w_syls) > max_syls:
        return "་".join(w_syls[:max_syls]) + "་"
    return whisper_uni


def transcribe_bo_hybrid_openpecha_vertex(audio, bo_blocks, asr_models, align_models, device, vertex_cfg, bo_slice_sec=9.0):
    """雙聲學模型融合 + 智慧呼吸停頓切點 + BoKenlm 音節語言模型 + Vertex LLM 校對翻譯模式 (hybrid_openpecha_vertex)：
    - 智慧呼吸停頓切點 (`split_audio_at_pauses`)：在 [6.0s, 11.0s] 內尋找 80ms 能量最低的換氣停頓點下刀，絕不切斷格西正在發音的藏文單詞。
    - Pass 1 (雙聲學模型聽寫 ModelFusion)：
      同時執行：
      (A) 1.28GB 的 `openpecha/op-wav2vec2-ft-v17` + `BoKenlm-syl-v0.4` CTC Beam Search 解碼（零幻覺、零循環，完整保留拉薩音每個音節並完成音節級正字校正）
      (B) CTranslate2 藏語 Whisper 模型 (`BO_ASR_MODEL`) 解碼，並用 CTC 的真實音節數截除 Whisper 在 30 秒補零區的句尾幻覺。
    - Pass 2 (佛典正字與中譯)：
      將 (A) 與 (B) 兩份互補的藏文聲學初稿連同《廣論》詞彙表送交 LLM (Vertex AI Gemini)，
      交叉比對還原出 100% 精準的標準藏文正字 (`text_bo`)，並直譯為流暢繁體中文 (`text_zh`)。
    """
    if not bo_blocks:
        return []

    ewts = globals().get("EWTS_CONVERTER") or pyewts.pyewts()
    bo_whisper = (asr_models or {}).get("bo")
    bo_align_tuple = (align_models or {}).get("bo")

    segments_for_pass2 = []
    for block in bo_blocks:
        blk_st = float(block["start"])
        blk_ed = float(block["end"])
        s1 = int(blk_st * SAMPLE_RATE)
        s2 = int(blk_ed * SAMPLE_RATE)
        blk_audio = audio[s1:s2]
        if len(blk_audio) < int(0.4 * SAMPLE_RATE):
            continue

        pause_slices = split_audio_at_pauses(
            blk_audio,
            sample_rate=SAMPLE_RATE,
            target_sec=bo_slice_sec,
            min_sec=6.0,
            max_sec=11.0,
            tail_merge_sec=1.8,
        )
        sub_chunks = [blk_audio[s_off:end_off] for s_off, end_off in pause_slices]
        if isinstance(bo_whisper, MPSWhisperPipeline):
            batch_raw_txts = bo_whisper.transcribe_batch(sub_chunks)
        else:
            batch_raw_txts = None

        for idx_s, (s_off, end_off) in enumerate(pause_slices):
            sub_chunk = sub_chunks[idx_s]
            st_sec = round(blk_st + s_off / float(SAMPLE_RATE), 3)
            ed_sec = round(blk_st + end_off / float(SAMPLE_RATE), 3)

            ctc_uni = decode_wav2vec2_ctc(sub_chunk, bo_align_tuple, device=device) if bo_align_tuple else ""

            whisper_uni = ""
            if bo_whisper is not None:
                if batch_raw_txts is not None:
                    raw_txt = (batch_raw_txts[idx_s] or "").strip()
                else:
                    sub_segs = _transcribe_single_slice(bo_whisper, sub_chunk, "bo")
                    raw_txt = " ".join((s.get("text") or "").strip() for s in sub_segs if (s.get("text") or "").strip()).strip()
                if raw_txt:
                    cleaned = clean_wylie_repetitions(raw_txt) if raw_txt.isascii() else raw_txt
                    if cleaned.isascii():
                        try:
                            whisper_uni = ewts.toUnicode(cleaned).strip()
                        except Exception:
                            whisper_uni = cleaned
                    else:
                        whisper_uni = cleaned

            whisper_gated = truncate_by_ctc_syllables(whisper_uni, ctc_uni) if (whisper_uni and ctc_uni) else whisper_uni
            primary_bo = whisper_gated or ctc_uni
            if not primary_bo:
                continue

            segments_for_pass2.append({
                "start": st_sec,
                "end": ed_sec,
                "start_time": f"{int(st_sec // 60):02d}:{int(st_sec % 60):02d}",
                "end_time": f"{int(ed_sec // 60):02d}:{int(ed_sec % 60):02d}",
                "tibetan_text": primary_bo,
                "ctc_tibetan_text": ctc_uni,
                "text": primary_bo,
                "translation_zh": "",
            })

    if not segments_for_pass2:
        return []

    if vertex_cfg and vertex_cfg.get("enabled"):
        reference_text = str(vertex_cfg.get("reference_text", "") or "")
        rolling_context = str(vertex_cfg.get("_rolling_context", "") or "")
        translate_tibetan_segments_to_zh(
            segments=segments_for_pass2,
            project=vertex_cfg["project"],
            location=vertex_cfg["location"],
            api_key=vertex_cfg.get("api_key", ""),
            model_name=vertex_cfg["model_name"],
            reference_text=reference_text,
            previous_context=rolling_context,
            thinking_level=vertex_cfg.get("thinking_level", "HIGH"),
            retry_attempts=vertex_cfg.get("retry_attempts", 5),
            retry_initial_delay_sec=vertex_cfg.get("retry_initial_delay_sec", 5.0),
            retry_backoff_multiplier=vertex_cfg.get("retry_backoff_multiplier", 2.0),
            retry_max_delay_sec=vertex_cfg.get("retry_max_delay_sec", 60.0),
            min_request_interval_sec=vertex_cfg.get("min_request_interval_sec", 2.0),
        )

    out_rows = []
    for seg in segments_for_pass2:
        text_bo = (seg.get("tibetan_text") or seg.get("text") or "").strip()
        text_zh = (seg.get("translation_zh") or "").strip()
        out_rows.append(build_transcript_row(
            seg["start"],
            seg["end"],
            "bo",
            text=text_zh or text_bo,
            text_bo=text_bo,
            text_zh=text_zh,
            source="hybrid_openpecha_vertex",
        ))

    out_rows.sort(key=lambda x: (x["start"], x["end"]))
    if out_rows and vertex_cfg is not None:
        vertex_cfg["_rolling_context"] = _format_rolling_context(out_rows)
    return out_rows

def transcribe_bo_with_vertex(audio, audio_path, bo_blocks, vertex_cfg, asr_models=None, align_models=None, device="cpu"):
    if not bo_blocks:
        return []

    mode = str(vertex_cfg.get("mode") or "hybrid_openpecha_vertex").lower()
    if mode not in {"hybrid_openpecha_vertex", "auto", "mixed_audio", "bo_blocks"}:
        raise ValueError(f"Unsupported VERTEX_BO_MODE: {mode}")

    if mode == "hybrid_openpecha_vertex":
        if not asr_models or asr_models.get("bo") is None:
            raise ValueError("hybrid_openpecha_vertex requires asr_models['bo'] (openpecha model) to be loaded.")
        return transcribe_bo_hybrid_openpecha_vertex(
            audio, bo_blocks, asr_models, align_models or {}, device, vertex_cfg
        )

    if mode in {"auto", "mixed_audio"}:
        try:
            mixed_rows = transcribe_bo_with_vertex_mixed_cut(audio_path, bo_blocks, vertex_cfg)
        except Exception as exc:
            print(
                f"[WARN] Vertex mixed-audio failed for {os.path.basename(audio_path)}: {exc}. "
                "Fallback to bo_blocks."
            )
        else:
            bo_total = total_duration(bo_blocks)
            mixed_total = total_duration(mixed_rows)
            coverage = mixed_total / bo_total if bo_total > 0 else 0.0
            min_coverage = float(vertex_cfg.get("mixed_min_coverage", 0.75))
            if mixed_rows and coverage >= min_coverage:
                print(
                    f"[INFO] Vertex mixed-audio accepted for {os.path.basename(audio_path)} "
                    f"(coverage={coverage:.1%})."
                )
                return mixed_rows

            reason = (
                "returned no Tibetan rows"
                if not mixed_rows
                else f"coverage {coverage:.1%} < {min_coverage:.1%}"
            )
            print(
                f"[WARN] Vertex mixed-audio {reason} for {os.path.basename(audio_path)}. "
                "Fallback to bo_blocks to prevent missing Tibetan segments."
            )

    return transcribe_bo_blocks_with_vertex(audio, audio_path, bo_blocks, vertex_cfg)


def transcribe_and_align(audio, asr_blocks, asr_models, align_models, device):
    rows = []
    for block in asr_blocks:
        lang = block["lang"]
        s1 = int(block["start"] * SAMPLE_RATE)
        s2 = int(block["end"] * SAMPLE_RATE)
        audio_chunk = audio[s1:s2]
        if len(audio_chunk) < int(0.4 * SAMPLE_RATE):
            continue

        base_segments = transcribe_block_fast(asr_models[lang], audio_chunk, lang)
        if not base_segments:
            continue

        aligned_segments = base_segments
        if align_models.get(lang) is not None:
            model_a, metadata = align_models[lang]
            try:
                aligned = whisperx.align(
                    base_segments,
                    model_a,
                    metadata,
                    audio_chunk,
                    device,
                    return_char_alignments=False,
                )
                aligned_segments = aligned.get("segments", base_segments)
            except Exception:
                aligned_segments = base_segments

        for s in aligned_segments:
            text = (s.get("text") or "").strip()
            if not text:
                continue
            st = round(block["start"] + float(s["start"]), 3)
            en = round(block["start"] + float(s["end"]), 3)
            if en <= st:
                continue
            rows.append(build_transcript_row(
                st,
                en,
                lang,
                text=text,
                text_zh=text if lang == "zh" else "",
                source="whisperx",
            ))

    rows.sort(key=lambda x: (x["start"], x["end"]))
    return rows


def parse_audio_cut_path(audio_path):
    stem = os.path.splitext(os.path.basename(audio_path))[0]
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected audio cut name: {audio_path}")
    chunk_index, chunk_start, chunk_end = parts[:3]
    return {
        "audio_path": audio_path,
        "audio_file": os.path.basename(audio_path),
        "chunk_index": int(chunk_index),
        "chunk_start": float(chunk_start),
        "chunk_end": float(chunk_end),
    }


def shift_time_rows(rows, offset_sec, extra_fields=None):
    shifted = []
    for row in rows:
        item = row.copy()
        if "start" in item:
            item["start"] = round(float(item["start"]) + offset_sec, 3)
        if "end" in item:
            item["end"] = round(float(item["end"]) + offset_sec, 3)
        if extra_fields:
            item.update(extra_fields)
        shifted.append(item)
    return shifted


def run_pipeline_for_audio_cut(audio_path, zh_model, asr_models, align_models, device, vertex_cfg=None):
    chunk_info = parse_audio_cut_path(audio_path)
    audio = whisperx.load_audio(audio_path)
    audio_duration_sec = len(audio) / float(SAMPLE_RATE)

    # 1. 細粒度 VAD（chunk_size=8 秒）+ 區段內中藏切換點自動切分
    raw_vad_segments = run_vad_segments(
        audio,
        zh_model.vad_model,
        chunk_size=8,
        vad_onset=0.45,
        vad_offset=0.30,
    )
    lid_pipeline = zh_model
    vad_segments = split_vad_segments_on_lang_change(
        lid_pipeline,
        audio,
        raw_vad_segments,
        min_split_sec=6.0,
        sub_win_sec=3.0,
        sub_hop_sec=1.5,
    )

    # 2. 高召回中藏 LID 偵測 + 孤島平滑校正
    lid_blocks = build_lid_blocks(vad_segments, min_lid_sec=1.8, max_gap=0.3, max_block_sec=8.0)

    seg_lang = ["other"] * len(vad_segments)
    seg_conf = [0.0] * len(vad_segments)

    for block in lid_blocks:
        lang, conf, _ = detect_block_language_zh_bo(
            lid_pipeline, audio, block["start"], block["end"]
        )
        for idx in block["idxs"]:
            seg_lang[idx] = lang
            seg_conf[idx] = conf

    seg_lang = smooth_langs_binary(
        seg_lang,
        seg_conf=seg_conf,
        vad_segments=vad_segments,
        default_lang="bo",
    )

    timeline_rows = []
    for i, (seg, lang) in enumerate(zip(vad_segments, seg_lang)):
        timeline_rows.append({
            "start": round(seg["start"], 3),
            "end": round(seg["end"], 3),
            "duration": round(seg["end"] - seg["start"], 3),
            "lang": lang,
            "lid_conf": round(seg_conf[i], 3),
        })

    # 3. 語言專屬區段合併 + 藏語區塊前後邊界安全緩衝 (0.5s~0.6s)
    bo_mode = str((vertex_cfg or {}).get("mode") or "hybrid_openpecha_vertex").lower()
    bo_target_max_sec = 10.0 if bo_mode == "hybrid_openpecha_vertex" else 35.0
    bo_target_gap_sec = 0.8 if bo_mode == "hybrid_openpecha_vertex" else 2.5
    local_asr_blocks = merge_for_asr(
        vad_segments,
        seg_lang,
        merge_gap=0.6,
        min_asr_sec=0.8,
        max_block_sec=40.0,
        bo_merge_gap=bo_target_gap_sec,
        bo_min_asr_sec=0.25,
        bo_max_block_sec=bo_target_max_sec,
    )
    zh_asr_blocks = [row for row in local_asr_blocks if row["lang"] == "zh"]
    bo_asr_blocks = pad_bo_blocks(
        [row for row in local_asr_blocks if row["lang"] == "bo"],
        local_asr_blocks,
        audio_duration_sec=audio_duration_sec,
        pad_start_sec=0.5,
        pad_end_sec=0.6,
    )

    transcribe_zh = bool(globals().get("TRANSCRIBE_ZH_INTERPRETER", True)) if vertex_cfg is None else bool(vertex_cfg.get("transcribe_zh_interpreter", globals().get("TRANSCRIBE_ZH_INTERPRETER", True)))
    transcript_rows = []
    if transcribe_zh and zh_asr_blocks:
        transcript_rows.extend(transcribe_and_align(audio, zh_asr_blocks, asr_models, align_models, device))

    if vertex_cfg and vertex_cfg.get("enabled") and bo_asr_blocks:
        transcript_rows.extend(transcribe_bo_with_vertex(audio, audio_path, bo_asr_blocks, vertex_cfg, asr_models=asr_models, align_models=align_models, device=device))
    elif bo_asr_blocks:
        transcript_rows.extend(transcribe_and_align(audio, bo_asr_blocks, asr_models, align_models, device))

    transcript_rows.sort(key=lambda x: (x["start"], x["end"]))

    extra_fields = {
        "chunk_index": chunk_info["chunk_index"],
        "audio_file": chunk_info["audio_file"],
    }
    timeline_rows = shift_time_rows(timeline_rows, chunk_info["chunk_start"], extra_fields)
    asr_blocks = shift_time_rows(local_asr_blocks, chunk_info["chunk_start"], extra_fields)
    transcript_rows = shift_time_rows(transcript_rows, chunk_info["chunk_start"], extra_fields)

    summary = {
        "chunk_index": chunk_info["chunk_index"],
        "audio_file": chunk_info["audio_file"],
        "chunk_start": chunk_info["chunk_start"],
        "chunk_end": chunk_info["chunk_end"],
        "vad_segments": len(vad_segments),
        "asr_blocks": len(asr_blocks),
        "transcript_rows": len(transcript_rows),
        "zh_segments": sum(1 for row in timeline_rows if row["lang"] == "zh"),
        "bo_segments": sum(1 for row in timeline_rows if row["lang"] == "bo"),
        "transcribe_zh_interpreter": transcribe_zh,
    }

    result = dict(chunk_info)
    result["summary"] = summary
    result["timeline_rows"] = timeline_rows
    result["asr_blocks"] = asr_blocks
    result["transcript_rows"] = transcript_rows
    return result


def sec_to_srt_time(sec):
    sec = max(0.0, float(sec))
    whole = int(sec)
    ms = int(round((sec - whole) * 1000))
    if ms == 1000:
        whole += 1
        ms = 0
    s = whole % 60
    m = (whole // 60) % 60
    h = whole // 3600
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def merge_text_piece(left, right, lang):
    left = (left or "").strip()
    right = (right or "").strip()
    if not left:
        return right
    if not right:
        return left
    sep = "" if lang in {"zh", "bo"} else " "
    return (left + sep + right).strip()


def merge_transcript_rows(rows, zh_max_gap=1.2, zh_max_block_sec=90.0):
    if not rows:
        return []
    out = [rows[0].copy()]
    for cur in rows[1:]:
        prev = out[-1]
        same_lang = cur["lang"] == prev["lang"]
        same_source = cur.get("source") == prev.get("source")
        same_chunk = (
            cur.get("chunk_index") == prev.get("chunk_index")
            and cur.get("audio_file") == prev.get("audio_file")
        )
        gap = cur["start"] - prev["end"]
        new_len = cur["end"] - prev["start"]
        should_merge_zh = (
            cur["lang"] == "zh"
            and same_lang
            and same_source
            and same_chunk
            and gap <= zh_max_gap
            and new_len <= zh_max_block_sec
        )
        if should_merge_zh:
            prev["end"] = cur["end"]
            prev["text"] = merge_text_piece(prev.get("text"), cur.get("text"), "zh")
            prev["text_zh"] = merge_text_piece(prev.get("text_zh"), cur.get("text_zh"), "zh")
        else:
            out.append(cur.copy())
    return out


def merge_subtitle_rows(rows, zh_max_gap=0.5, zh_max_block_sec=12.0, bo_max_gap=0.8, bo_max_block_sec=12.0):
    if not rows:
        return []
    out = [rows[0].copy()]
    for cur in rows[1:]:
        prev = out[-1]
        same_lang = cur["lang"] == prev["lang"]
        same_source = cur.get("source") == prev.get("source")
        same_chunk = (
            cur.get("chunk_index") == prev.get("chunk_index")
            and cur.get("audio_file") == prev.get("audio_file")
        )
        gap = cur["start"] - prev["end"]
        new_len = cur["end"] - prev["start"]
        if cur["lang"] == "zh":
            allowed_gap = zh_max_gap
            allowed_len = zh_max_block_sec
        elif cur["lang"] == "bo":
            allowed_gap = bo_max_gap
            allowed_len = bo_max_block_sec
        else:
            allowed_gap = -1.0
            allowed_len = -1.0
        if same_lang and same_source and same_chunk and gap <= allowed_gap and new_len <= allowed_len:
            prev["end"] = cur["end"]
            prev["text"] = merge_text_piece(prev.get("text"), cur.get("text"), cur["lang"])
            prev["text_bo"] = merge_text_piece(prev.get("text_bo"), cur.get("text_bo"), "bo")
            prev["text_zh"] = merge_text_piece(prev.get("text_zh"), cur.get("text_zh"), "zh")
        else:
            out.append(cur.copy())
    return out


def write_csv(rows, path, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


def write_srt(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows, 1):
            text_bo = (r.get("text_bo") or "").strip()
            text_zh = (r.get("text_zh") or r.get("text") or "").strip()
            f.write(f"{i}\n")
            f.write(f"{sec_to_srt_time(r['start'])} --> {sec_to_srt_time(r['end'])}\n")
            if r["lang"] == "bo":
                if text_bo:
                    f.write(f"[藏文] {text_bo}\n")
                if text_zh:
                    f.write(f"[中譯] {text_zh}\n")
            else:
                f.write(f"[漢語] {text_zh}\n")
            f.write("\n")


def sec_to_doc_time(sec):
    total_sec = max(0, int(round(float(sec))))
    s = total_sec % 60
    m = (total_sec // 60) % 60
    h = total_sec // 3600
    return f"{h}:{m:02d}:{s:02d}"


def rtf_escape(text):
    out = []
    for ch in str(text or ""):
        code = ord(ch)
        if ch == "\\":
            out.append(r"\\")
        elif ch == "{":
            out.append(r"\{")
        elif ch == "}":
            out.append(r"\}")
        elif ch == "\n":
            out.append(r"\line ")
        elif 32 <= code <= 126:
            out.append(ch)
        else:
            signed = code if code <= 32767 else code - 65536
            out.append(f"\\u{signed}")
    return "".join(out)


def build_rtf_sections(rows, zh_max_gap=1.2, zh_max_block_sec=90.0, bo_max_gap=1.2, bo_max_block_sec=90.0):
    sections = []
    current = None
    for row in rows:
        lang = row.get("lang")
        if lang not in {"zh", "bo"}:
            continue
        if lang == "bo":
            has_content = bool((row.get("text_bo") or "").strip() or (row.get("text_zh") or "").strip())
            allowed_gap = bo_max_gap
            allowed_len = bo_max_block_sec
        else:
            has_content = bool((row.get("text_zh") or row.get("text") or "").strip())
            allowed_gap = zh_max_gap
            allowed_len = zh_max_block_sec
        if not has_content:
            continue
        item = row.copy()
        if current is None:
            current = {
                "lang": lang,
                "source": item.get("source"),
                "chunk_index": item.get("chunk_index"),
                "audio_file": item.get("audio_file"),
                "start": item["start"],
                "end": item["end"],
                "rows": [item],
            }
            continue
        same_lang = lang == current["lang"]
        same_source = item.get("source") == current.get("source")
        same_chunk = (
            item.get("chunk_index") == current.get("chunk_index")
            and item.get("audio_file") == current.get("audio_file")
        )
        gap = float(item["start"]) - float(current["end"])
        new_len = float(item["end"]) - float(current["start"])
        if same_lang and same_source and same_chunk and gap <= allowed_gap and new_len <= allowed_len:
            current["end"] = item["end"]
            current["rows"].append(item)
        else:
            sections.append(current)
            current = {
                "lang": lang,
                "source": item.get("source"),
                "chunk_index": item.get("chunk_index"),
                "audio_file": item.get("audio_file"),
                "start": item["start"],
                "end": item["end"],
                "rows": [item],
            }
    if current is not None:
        sections.append(current)
    return sections


def write_rtf_sections(sections, path):
    lines = [
        r"{\rtf1\ansi\deff0\uc0",
        r"{\fonttbl{\f0\fnil Helvetica;}}",
        r"\f0\fs24",
    ]
    for section in sections:
        title = f"{sec_to_doc_time(section['start'])}-{sec_to_doc_time(section['end'])}"
        lines.append(rtf_escape(title) + r"\line")
        if section["lang"] == "bo":
            for idx, row in enumerate(section["rows"]):
                text_bo = (row.get("text_bo") or "").strip()
                text_zh = (row.get("text_zh") or "").strip()
                if text_bo:
                    lines.append(rtf_escape(f"藏文：{text_bo}") + r"\line")
                if text_zh:
                    lines.append(rtf_escape(f"漢譯：{text_zh}") + r"\line")
                if idx != len(section["rows"]) - 1:
                    lines.append(r"\line")
        else:
            for row in section["rows"]:
                text_zh = (row.get("text_zh") or row.get("text") or "").strip()
                if text_zh:
                    lines.append(rtf_escape(text_zh) + r"\line")
        lines.append(r"\line\line")
    lines.append("}")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def build_unicode_transcript_rows(rows, ewts_converter):
    unicode_rows = []
    for row in rows:
        item = row.copy()
        text_bo = (item.get("text_bo") or "").strip()
        if item.get("lang") == "bo" and text_bo and text_bo.isascii():
            try:
                item["text_bo"] = ewts_converter.toUnicode(text_bo)
            except Exception:
                item["text_bo"] = text_bo
        unicode_rows.append(item)
    return unicode_rows


def convert_rows_zh_to_traditional(rows, zh_converter):
    converted_rows = []
    for row in rows:
        item = row.copy()
        original_text = (item.get("text") or "")
        original_text_zh = (item.get("text_zh") or "")
        if original_text_zh:
            item["text_zh"] = zh_converter.convert(original_text_zh)
        if item.get("lang") == "zh":
            target = original_text if original_text else item.get("text_zh", "")
            if target:
                item["text"] = zh_converter.convert(target)
        elif item.get("lang") == "bo" and original_text_zh and original_text.strip() == original_text_zh.strip():
            item["text"] = item["text_zh"]
        converted_rows.append(item)
    return converted_rows


def checkpoint_path_for_audio(audio_path, checkpoint_dir):
    chunk_info = parse_audio_cut_path(audio_path)
    stem = Path(audio_path).stem
    filename = f"{chunk_info['chunk_index']:04d}_{stem}.json"
    return os.path.join(checkpoint_dir, filename)


def save_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_chunk_checkpoint(result, checkpoint_dir):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = checkpoint_path_for_audio(result["audio_path"], checkpoint_dir)
    save_json(path, result)
    return path


def load_chunk_checkpoint(audio_path, checkpoint_dir):
    path = checkpoint_path_for_audio(audio_path, checkpoint_dir)
    if not os.path.exists(path):
        return None, path
    return load_json(path), path


def write_checkpoint_manifest(chunk_results, checkpoint_dir):
    manifest_path = os.path.join(checkpoint_dir, "manifest.json")
    payload = {
        "total_chunks": len(chunk_results),
        "chunk_summaries": [result.get("summary", {}) for result in chunk_results],
    }
    save_json(manifest_path, payload)
    return manifest_path

## 3. 模型與執行參數

這裡集中設定本次執行會用到的音檔、輸出資料夾，以及中文本地 ASR / alignment 與藏語 Vertex AI 轉寫設定。

常用調整項目：
- `AUDIO_PATHS`：要批次處理的所有切片，預設就是 `audio_cuts`。
- `OUT_DIR`：所有輸出檔的目錄。
- `DEVICE`、`COMPUTE_TYPE`：依 CPU / GPU 環境切換；目前 notebook 仍以 `cpu` 為主。
- `USE_VERTEXAI_FOR_BO`：是否把藏語區段改交給 Vertex AI。
- `VERTEX_BO_MODE`：`auto` 會先嘗試整段漢藏混音切片，再於結果不足時退回 `bo_blocks`；也可手動指定 `mixed_audio` 或 `bo_blocks`。
- `VERTEX_PROJECT`、`VERTEX_LOCATION`、`VERTEX_STAGING_GCS_PREFIX`：Vertex AI 與臨時 GCS staging 所需設定。
- `VERTEX_MODEL_NAME`：要用的 Vertex Gemini 多模態模型。
- `VERTEX_THINKING_LEVEL`：Gemini 3 的思考強度，這裡預設使用 `HIGH`。
- `VERTEX_MAX_OUTPUT_TOKENS`：控制藏語段落 JSON 輸出的 token 上限；遇到回傳被截斷時可往上調。
- `VERTEX_RETRY_ATTEMPTS`、`VERTEX_RETRY_INITIAL_DELAY_SEC`、`VERTEX_RETRY_BACKOFF_MULTIPLIER`、`VERTEX_RETRY_MAX_DELAY_SEC`、`VERTEX_REQUEST_PAUSE_SEC`：控制 Vertex 429/5xx 的退避重試與請求節流。
- `CHECKPOINT_DIR`、`RESUME_FROM_CHECKPOINT`、`FORCE_REPROCESS_CHUNKS`：控制每個切片是否即時落盤，以及重跑時要續跑還是強制重算。
- `VERTEX_MIXED_MIN_COVERAGE`：`auto` 模式下，整段混音結果至少要覆蓋多少比例的藏語區塊才視為可接受。
- `ZH_ASR_MODEL`、`BO_ASR_MODEL`：中文與藏文的轉寫模型；若啟用 Vertex，`BO_ASR_MODEL` 主要只保留為本地備援。
- `ZH_ALIGN_MODEL`、`BO_ALIGN_MODEL`：需要更準時間戳時使用的對齊模型；若啟用 Vertex，藏語不會再走本地 alignment。


In [7]:
# 如果尚未安裝：
# !pip install whisperx pyannote.audio google-genai google-cloud-storage opencc-python-reimplemented

import io
import os
from pathlib import Path
import warnings
import wave
import logging
import csv
import json
import opencc
import ctypes
import glob as _glob
import pyewts
import torch

# 預先載入 macOS 下 kenlm 所需的 liblzma 動態庫
for _lzma_p in [
    "/Users/pfhsu/miniforge3/lib/liblzma.5.dylib",
    *_glob.glob("/Users/pfhsu/miniforge3/pkgs/liblzma-*/lib/liblzma.5.dylib"),
]:
    if os.path.exists(_lzma_p):
        try:
            ctypes.CDLL(_lzma_p, mode=ctypes.RTLD_GLOBAL)
            break
        except Exception:
            pass

from vertexai_tibetan_stt import (
    DEFAULT_LAMRIM_GLOSSARY,
    build_gcs_uri,
    delete_gcs_object,
    make_staging_object_name,
    timestamp_to_seconds,
    transcribe_tibetan_audio_from_gcs,
    translate_tibetan_segments_to_zh,
    upload_bytes_to_gcs,
    upload_file_to_gcs,
)

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("XDG_CACHE_HOME", str((Path.cwd() / ".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

warnings.filterwarnings(
    "ignore",
    message=r"\s*torchcodec is not installed correctly so built-in audio decoding will fail\.",
    module=r"pyannote\.audio\.core\.io",
)
warnings.filterwarnings(
    "ignore",
    message=r"torchaudio\._backend\.list_audio_backends has been deprecated\..*",
    module=r"speechbrain\.utils\.torch_audio_backend",
)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.ERROR)

import whisperx
from whisperx.audio import SAMPLE_RATE, N_SAMPLES, log_mel_spectrogram
from whisperx.vads import Vad, Pyannote

# 如果沒有在 shell 先 export，可取消註解後填入自己的值：
os.environ["GOOGLE_CLOUD_PROJECT"] = "gen-lang-client-0883795355"
# os.environ["VERTEX_LOCATION"] = "global"
# os.environ["VERTEX_STAGING_GCS_PREFIX"] = "gs://webuddhist/vertex-staging"
# os.environ["GOOGLE_CLOUD_API_KEY"] = "your-google-cloud-api-key"  # 可選，express mode 可用

AUDIO_PATHS = audio_cuts
OUT_DIR = "./out"

DEVICE = "cpu"                  # Mac 建議先 "cpu", 有gpu 設 "cuda"
COMPUTE_TYPE = "int8"           # cpu 設定 "int8", gpu 設定 "float16"
USE_VERTEXAI_FOR_BO = True
VERTEX_BO_MODE = "hybrid_openpecha_vertex"  # 預設最強模式：Pass 1 用本地 OpenPecha 藏語聲學模型聽音節 -> Pass 2 用 Vertex LLM 校對正字與翻成中文（可改 "bo_blocks"）
VERTEX_TWO_PASS = True          # 兩階段處理：Pass 1 純藏文聽寫 -> Pass 2 結合前後文與廣論詞彙表精譯中文

# 讀取最開頭 Cell 1 設定的 TRANSCRIBE_ZH_INTERPRETER 開關（若未執行 Cell 1 則預設為 True）
TRANSCRIBE_ZH_INTERPRETER = bool(globals().get("TRANSCRIBE_ZH_INTERPRETER", True))

# 可選：填入本堂課《菩提道次第廣論》進度原文或專有名詞（會自動附加於內建詞彙表後，大幅減少引經與同音字誤判）
LAMRIM_REFERENCE_TEXT = """
本堂課進度：《菩提道次第廣論》卷一・第 1 頁（皈敬頌、造者殊勝、法殊勝、講聞軌理）
- 皈敬頌（མཆོད་བརྗོད）：南無姑如曼殊廓喀耶（ན་མོ་གུ་རུ་མཉྫུ་གྷོ་ཥཱ་ཡ། 頂禮上師妙吉祥音）
- 俱胝圓滿妙善所生身，成滿無邊眾生希願語，如實觀見無餘所知意，怙主無上釋迦尊前禮。
- 佛陀（སངས་རྒྱས）、薄伽梵（བཅོམ་ལྡན་འདས）、釋迦牟尼佛（ཤཱཀྱ་ཐུབ་པ）
- 文殊師利菩薩（འཇམ་དཔལ་དབྱངས）、慈氏彌勒菩薩（བྱམས་པ་མགོན་པོ）
- 龍樹菩薩（ཀླུ་སྒྲུབ）、聖勇/無著菩薩（ཐོགས་མེད）、月稱菩薩（ཟླ་བ་གྲགས་པ）、世親菩薩（དབྱིག་གཉེན）
- 阿底峽尊者（ཇོ་བོ་རྗེ་དཔལ་ལྡན་ཨ་ཏི་ཤ / མར་མེ་མཛད་དཔལ་ཡེ་ཤེས）、宗喀巴大師（རྗེ་ཙོང་ཁ་པ）
- 造者殊勝（མཛད་པ་པོ་ཆེ་བའི་ཡོན་ཏན）、法殊勝（ཆོས་ཀྱི་ཆེ་བའི་ཡོན་ཏན）、如何講聞相應正法（ཆོས་ཇི་ལྟར་ཉན་བཤད་བྱ་བའི་ཚུལ）
""".strip()

CHECKPOINT_DIR = os.path.join(OUT_DIR, "chunk_checkpoints")
RESUME_FROM_CHECKPOINT = True
FORCE_REPROCESS_CHUNKS = False   # 升級為 bo_blocks + Two-Pass 後，首次重跑請設為 True 以覆蓋舊的缺漏 checkpoint
VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT", "gen-lang-client-0883795355")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "global")
# 自動載入本地 .env（若存在），避免將 API Key 寫死在公開 Git 儲存庫中
_env_file = Path.cwd() / ".env"
if _env_file.exists():
    for _line in _env_file.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip("\"'"))
VERTEX_API_KEY = os.getenv("GOOGLE_CLOUD_API_KEY", "")
VERTEX_MODEL_NAME = os.getenv("VERTEX_MODEL_NAME", "gemini-3.1-pro-preview")
VERTEX_STAGING_GCS_PREFIX = os.getenv("VERTEX_STAGING_GCS_PREFIX", "gs://webuddhist/vertex-staging")
VERTEX_THINKING_LEVEL = os.getenv("VERTEX_THINKING_LEVEL", "HIGH")
VERTEX_MAX_OUTPUT_TOKENS = int(os.getenv("VERTEX_MAX_OUTPUT_TOKENS", "65535"))
VERTEX_RETRY_ATTEMPTS = int(os.getenv("VERTEX_RETRY_ATTEMPTS", "5"))
VERTEX_RETRY_INITIAL_DELAY_SEC = float(os.getenv("VERTEX_RETRY_INITIAL_DELAY_SEC", "5"))
VERTEX_RETRY_BACKOFF_MULTIPLIER = float(os.getenv("VERTEX_RETRY_BACKOFF_MULTIPLIER", "2"))
VERTEX_RETRY_MAX_DELAY_SEC = float(os.getenv("VERTEX_RETRY_MAX_DELAY_SEC", "60"))
VERTEX_REQUEST_PAUSE_SEC = float(os.getenv("VERTEX_REQUEST_PAUSE_SEC", "2"))
VERTEX_CLEANUP_STAGED_OBJECTS = True
VERTEX_MIXED_MIN_COVERAGE = 0.75

ZH_ASR_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"
# 藏語 Whisper 聲學模型（CTranslate2 格式）：
# - "openpecha_op-whisper_small-ft-v2"：OpenPecha/Monlam 佛法與多方言微調模型（搭配下方 BO_ALIGN_MODEL 1.28GB Wav2Vec2-CTC 雙模型融合效果最佳）
# - "nict_tib1_lhasa_whisper_small_ct2"：NICT-Tib1 拉薩音語料庫微調模型（已由 ct2-transformers-converter 轉換好，可直接切換）
BO_ASR_MODEL = "openpecha_op-whisper_small-ft-v2"
# 若為 Apple Silicon (M1/M2/M3/M4)，預設啟用 PyTorch MPS GPU 批次推論執行藏語 Whisper：
USE_MPS_FOR_BO_WHISPER = True
BO_MPS_WHISPER_MODEL = "openpecha/op-whisper_small-ft-v2"

ZH_ALIGN_MODEL = "jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn"
BO_ALIGN_MODEL = "openpecha/op-wav2vec2-ft-v17"
# OpenPecha BoKenlm-syl-v0.4 藏文音節 N-gram 語言模型（供 Wav2Vec2-CTC Beam Search 解碼使用）：
BO_KENLM_ARPA_PATH = str((Path.cwd() / ".cache" / "bokenlm_syl_v04_3gram.arpa").resolve())

if not AUDIO_PATHS:
    raise ValueError("No audio cuts found. Run the slicing cells first.")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
EWTS_CONVERTER = pyewts.pyewts()
ZH_TRAD_CONVERTER = opencc.OpenCC('s2tw.json')
VERTEX_CONFIG = {
    "enabled": USE_VERTEXAI_FOR_BO,
    "mode": VERTEX_BO_MODE,
    "two_pass": VERTEX_TWO_PASS,
    "transcribe_zh_interpreter": TRANSCRIBE_ZH_INTERPRETER,
    "reference_text": LAMRIM_REFERENCE_TEXT,
    "project": VERTEX_PROJECT,
    "location": VERTEX_LOCATION,
    "api_key": VERTEX_API_KEY,
    "model_name": VERTEX_MODEL_NAME,
    "staging_gcs_prefix": VERTEX_STAGING_GCS_PREFIX,
    "thinking_level": VERTEX_THINKING_LEVEL,
    "max_output_tokens": VERTEX_MAX_OUTPUT_TOKENS,
    "retry_attempts": VERTEX_RETRY_ATTEMPTS,
    "retry_initial_delay_sec": VERTEX_RETRY_INITIAL_DELAY_SEC,
    "retry_backoff_multiplier": VERTEX_RETRY_BACKOFF_MULTIPLIER,
    "retry_max_delay_sec": VERTEX_RETRY_MAX_DELAY_SEC,
    "min_request_interval_sec": VERTEX_REQUEST_PAUSE_SEC,
    "cleanup_staged_objects": VERTEX_CLEANUP_STAGED_OBJECTS,
    "mixed_min_coverage": VERTEX_MIXED_MIN_COVERAGE,
}

if VERTEX_CONFIG["enabled"]:
    missing = []
    if not VERTEX_CONFIG["staging_gcs_prefix"]:
        missing.append("staging_gcs_prefix")
    if not (VERTEX_CONFIG["project"] or VERTEX_CONFIG.get("api_key")):
        missing.append("VERTEX_PROJECT / GOOGLE_CLOUD_PROJECT or GOOGLE_CLOUD_API_KEY")
    if missing:
        raise ValueError(
            "USE_VERTEXAI_FOR_BO=True 時請先設定: "
            f"{', '.join(missing)}\n\n"
            "可在 notebook 這一格前面加入：\n"
            'os.environ["VERTEX_PROJECT"] = "your-gcp-project-id"\n'
            'os.environ["VERTEX_LOCATION"] = "global"\n'
            'os.environ["VERTEX_STAGING_GCS_PREFIX"] = "gs://your-bucket/vertex-staging"\n'
            'os.environ["GOOGLE_CLOUD_API_KEY"] = "your-google-cloud-api-key"  # 可選，express mode'
        )

In [8]:
zh_model = whisperx.load_model(
    ZH_ASR_MODEL,
    DEVICE,
    compute_type=COMPUTE_TYPE,
    language="zh",   # 用於 VAD 與中藏 LID；若 TRANSCRIBE_ZH_INTERPRETER=True 亦用於中文口譯轉寫
)

asr_models = {"zh": zh_model}
need_local_bo_model = (not USE_VERTEXAI_FOR_BO) or (str(VERTEX_BO_MODE).lower() == "hybrid_openpecha_vertex")
if need_local_bo_model:
    if globals().get("USE_MPS_FOR_BO_WHISPER", True) and torch.backends.mps.is_available():
        bo_model = MPSWhisperPipeline(
            model_id=globals().get("BO_MPS_WHISPER_MODEL", "openpecha/op-whisper_small-ft-v2"),
            device="mps",
        )
        print(f"bo Whisper model loaded on Apple M3 MPS GPU ({bo_model.model_id})")
    else:
        bo_model = whisperx.load_model(
            BO_ASR_MODEL,
            DEVICE,
            compute_type=COMPUTE_TYPE,
            language="bo",
        )
    asr_models["bo"] = bo_model
    if USE_VERTEXAI_FOR_BO:
        print(
            f"bo transcription will use HYBRID mode: Pass 1 OpenPecha ({BO_ASR_MODEL}) "
            f"-> Pass 2 Vertex AI ({VERTEX_MODEL_NAME}, thinking={VERTEX_THINKING_LEVEL})"
        )
else:
    bo_model = None
    print(f"bo transcription will use Vertex AI only: {VERTEX_MODEL_NAME} ({VERTEX_BO_MODE}, two_pass={VERTEX_TWO_PASS}, thinking={VERTEX_THINKING_LEVEL})")

align_models = {"zh": None, "bo": None}
align_targets = []
if TRANSCRIBE_ZH_INTERPRETER:
    align_targets.append(("zh", ZH_ALIGN_MODEL))
    print("TRANSCRIBE_ZH_INTERPRETER=True -> 將聽寫現場中文譯師（啟用中文 WhisperX + Align）")
else:
    print("TRANSCRIBE_ZH_INTERPRETER=False -> 略過現場中文譯師（不跑中文聽寫、不載入中文 Align 模型）")

if need_local_bo_model:
    align_targets.append(("bo", BO_ALIGN_MODEL))

for lang, model_name in align_targets:
    try:
        align_dev = "mps" if (lang == "bo" and torch.backends.mps.is_available()) else DEVICE
        align_models[lang] = whisperx.load_align_model(
            language_code=lang,
            device=align_dev,
            model_name=model_name,
        )
        print(f"{lang} align model loaded")
    except Exception as e:
        align_models[lang] = None
        print(f"[WARN] {lang} align unavailable:", e)

print(f"audio cuts to process: {len(AUDIO_PATHS)}")
print("first audio cut:", AUDIO_PATHS[0])

2026-08-09 19:01:58 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../miniforge3/envs/whisper-jax/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


bo transcription will use Vertex AI: gemini-3.1-pro-preview (mixed_audio, thinking=HIGH)


/Users/pfhsu/miniforge3/envs/whisper-jax/lib/python3.12/site-packages/transformers/configuration_utils.py:315: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


zh align model loaded
audio cuts to process: 15
first audio cut: audio_5min/0_0_300.mp3


## 4. 載入模型並批次執行流程

這個區塊會先把中文 ASR / alignment 模型載入一次，接著用 `for` 迴圈依序處理 `AUDIO_PATHS` 裡的每個 5 分鐘切片。
中文會固定保留在 `whisperx + align`，因為目前這條路對字幕時間軸通常比 Gemini / Vertex 音訊理解更穩定。

若 `USE_VERTEXAI_FOR_BO=True`，流程會維持原本的 VAD / LID，只是把藏語轉寫改成 Vertex AI，並要求模型回傳「藏語原文 + 最貼近字面意義的漢語直譯」：
- `auto`：先把整個 5 分鐘切片送去評估漢藏混音轉寫，只保留和 `bo` 區塊重疊的結果；若失敗或覆蓋率太低，會自動退回 `bo_blocks`。
- `mixed_audio`：強制整段切片送 Vertex，不做 fallback。
- `bo_blocks`：直接沿用原本切好的藏語區塊，一段段送 Vertex。

`run_vad_segments` 裡仍然會忽略短於 0.2 秒的片段，避免太短的噪音影響結果。

另外現在每個切片跑完都會即時寫入 `out/chunk_checkpoints/*.json`。如果中途失敗，重跑這個 cell 時會優先讀取已存在的 checkpoint，只補跑尚未完成的切片；若想全部重算，把 `FORCE_REPROCESS_CHUNKS=True` 即可。


In [9]:
chunk_results = []
fresh_count = 0
checkpoint_count = 0
manifest_path = os.path.join(CHECKPOINT_DIR, "manifest.json")

# 確保 VERTEX_CONFIG 與最開頭的 TRANSCRIBE_ZH_INTERPRETER 同步
VERTEX_CONFIG["transcribe_zh_interpreter"] = bool(TRANSCRIBE_ZH_INTERPRETER)

for audio_path in AUDIO_PATHS:
    checkpoint_result = None
    checkpoint_path = checkpoint_path_for_audio(audio_path, CHECKPOINT_DIR)
    if RESUME_FROM_CHECKPOINT and not FORCE_REPROCESS_CHUNKS:
        checkpoint_result, checkpoint_path = load_chunk_checkpoint(audio_path, CHECKPOINT_DIR)
        # 若現在要求聽寫中文譯師，但舊 checkpoint 當時沒跑中文聽寫，則自動重跑該切片
        if checkpoint_result is not None and TRANSCRIBE_ZH_INTERPRETER:
            ckpt_had_zh = checkpoint_result.get("summary", {}).get("transcribe_zh_interpreter", True)
            if not ckpt_had_zh:
                checkpoint_result = None

    if checkpoint_result is not None:
        result = checkpoint_result
        if not TRANSCRIBE_ZH_INTERPRETER:
            result["transcript_rows"] = [
                r for r in result.get("transcript_rows", []) if r.get("lang") != "zh"
            ]
            result["summary"]["transcript_rows"] = len(result["transcript_rows"])
        source = "checkpoint"
        checkpoint_count += 1
    else:
        result = run_pipeline_for_audio_cut(
            audio_path,
            zh_model=zh_model,
            asr_models=asr_models,
            align_models=align_models,
            device=DEVICE,
            vertex_cfg=VERTEX_CONFIG,
        )
        checkpoint_path = save_chunk_checkpoint(result, CHECKPOINT_DIR)
        source = "fresh"
        fresh_count += 1

    chunk_results.append(result)
    write_checkpoint_manifest(chunk_results, CHECKPOINT_DIR)

    summary = result["summary"]
    print(
        f"[{summary['chunk_index']}] {summary['audio_file']} | "
        f"VAD={summary['vad_segments']} | "
        f"ASR={summary['asr_blocks']} | "
        f"Transcript={summary['transcript_rows']} | "
        f"ZH={summary['zh_segments']} | "
        f"BO={summary['bo_segments']} | "
        f"zh_transcribed={TRANSCRIBE_ZH_INTERPRETER} | "
        f"source={source} | checkpoint={os.path.basename(checkpoint_path)}"
    )

chunk_summaries = [result["summary"] for result in chunk_results]
print("chunks processed:", len(chunk_results))
print("loaded from checkpoint:", checkpoint_count)
print("freshly processed:", fresh_count)
print("checkpoint manifest:", manifest_path)
chunk_summaries[:3]

INFO:faster_whisper:Processing audio with duration 00:05.552
INFO:faster_whisper:Processing audio with duration 00:28.772
INFO:faster_whisper:Processing audio with duration 00:42.221
INFO:faster_whisper:Processing audio with duration 00:41.192
INFO:faster_whisper:Processing audio with duration 00:44.803
INFO:google_genai._api_client:The user provided Vertex AI API key will take precedence over the project/location from the environment variables.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[0] 0_0_300.mp3 | VAD=13 | ASR=10 | Transcript=106 | ZH=8 | BO=5 | source=fresh | checkpoint=0000_0_0_300.json


INFO:faster_whisper:Processing audio with duration 00:29.987
INFO:faster_whisper:Processing audio with duration 00:42.356
INFO:faster_whisper:Processing audio with duration 00:40.416
INFO:faster_whisper:Processing audio with duration 00:41.580
INFO:faster_whisper:Processing audio with duration 00:10.749
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[1] 1_300_600.mp3 | VAD=14 | ASR=11 | Transcript=97 | ZH=8 | BO=6 | source=fresh | checkpoint=0001_1_300_600.json


INFO:faster_whisper:Processing audio with duration 00:26.949
INFO:faster_whisper:Processing audio with duration 00:29.987
INFO:faster_whisper:Processing audio with duration 00:21.516
INFO:faster_whisper:Processing audio with duration 00:28.519
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[2] 2_600_900.mp3 | VAD=11 | ASR=11 | Transcript=53 | ZH=4 | BO=7 | source=fresh | checkpoint=0002_2_600_900.json


... (25 outputs left out)

[12] 12_3600_3900.mp3 | VAD=12 | ASR=11 | Transcript=102 | ZH=5 | BO=7 | source=fresh | checkpoint=0012_12_3600_3900.json


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


[13] 13_3900_3903.mp3 | VAD=1 | ASR=1 | Transcript=1 | ZH=0 | BO=1 | source=fresh | checkpoint=0013_13_3900_3903.json
chunks processed: 15
loaded from checkpoint: 0
freshly processed: 15
checkpoint manifest: ./out/chunk_checkpoints/manifest.json


[{'chunk_index': 0,
  'audio_file': '0_0_300.mp3',
  'chunk_start': 0.0,
  'chunk_end': 300.0,
  'vad_segments': 13,
  'asr_blocks': 10,
  'transcript_rows': 106,
  'zh_segments': 8,
  'bo_segments': 5},
 {'chunk_index': 1,
  'audio_file': '1_300_600.mp3',
  'chunk_start': 300.0,
  'chunk_end': 600.0,
  'vad_segments': 14,
  'asr_blocks': 11,
  'transcript_rows': 97,
  'zh_segments': 8,
  'bo_segments': 6},
 {'chunk_index': 2,
  'audio_file': '2_600_900.mp3',
  'chunk_start': 600.0,
  'chunk_end': 900.0,
  'vad_segments': 11,
  'asr_blocks': 11,
  'transcript_rows': 53,
  'zh_segments': 4,
  'bo_segments': 7}]

## 5. 檢查語言時間軸（LID）

上一個 cell 已經把每個切片的 VAD / LID / ASR 都跑完，這裡開始把 LID 結果整理成同一條全域時間軸。

`timeline_rows` 會保留 `chunk_index` 與 `audio_file`，方便回頭定位是哪一個 5 分鐘切片；而 `start` / `end` 則已經換算成原始長音檔的時間。


In [10]:
timeline_rows = []
for result in chunk_results:
    timeline_rows.extend(result["timeline_rows"])

timeline_rows.sort(key=lambda x: (x["start"], x["end"]))
print("timeline rows:", len(timeline_rows))
print(chunk_summaries[:3])
timeline_rows[:10]


timeline rows: 171
[{'chunk_index': 0, 'audio_file': '0_0_300.mp3', 'chunk_start': 0.0, 'chunk_end': 300.0, 'vad_segments': 13, 'asr_blocks': 10, 'transcript_rows': 106, 'zh_segments': 8, 'bo_segments': 5}, {'chunk_index': 1, 'audio_file': '1_300_600.mp3', 'chunk_start': 300.0, 'chunk_end': 600.0, 'vad_segments': 14, 'asr_blocks': 11, 'transcript_rows': 97, 'zh_segments': 8, 'bo_segments': 6}, {'chunk_index': 2, 'audio_file': '2_600_900.mp3', 'chunk_start': 600.0, 'chunk_end': 900.0, 'vad_segments': 11, 'asr_blocks': 11, 'transcript_rows': 53, 'zh_segments': 4, 'bo_segments': 7}]


[{'start': 1.465,
  'end': 29.444,
  'duration': 27.979,
  'lang': 'bo',
  'lid_conf': 0.85,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 30.524,
  'end': 56.663,
  'duration': 26.139,
  'lang': 'bo',
  'lid_conf': 0.912,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 57.355,
  'end': 84.068,
  'duration': 26.713,
  'lang': 'bo',
  'lid_conf': 0.663,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 84.44,
  'end': 105.719,
  'duration': 21.279,
  'lang': 'bo',
  'lid_conf': 0.698,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 107.035,
  'end': 135.689,
  'duration': 28.654,
  'lang': 'bo',
  'lid_conf': 0.92,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 136.128,
  'end': 141.68,
  'duration': 5.552,
  'lang': 'zh',
  'lid_conf': 0.928,
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 142.979,
  'end': 171.751,
  'duration': 28.772,
  'lang': 'zh',
  'lid_conf': 0.998,
  'chunk_index': 0,
  

In [11]:
print(timeline_rows[:10])

# 1) 語言時間軸 CSV
timeline_csv = os.path.join(OUT_DIR, "speech_lang_timeline.csv")
write_csv(
    timeline_rows,
    timeline_csv,
    ["chunk_index", "audio_file", "start", "end", "duration", "lid_conf", "lang"],
)
timeline_csv


[{'start': 1.465, 'end': 29.444, 'duration': 27.979, 'lang': 'bo', 'lid_conf': 0.85, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 30.524, 'end': 56.663, 'duration': 26.139, 'lang': 'bo', 'lid_conf': 0.912, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 57.355, 'end': 84.068, 'duration': 26.713, 'lang': 'bo', 'lid_conf': 0.663, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 84.44, 'end': 105.719, 'duration': 21.279, 'lang': 'bo', 'lid_conf': 0.698, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 107.035, 'end': 135.689, 'duration': 28.654, 'lang': 'bo', 'lid_conf': 0.92, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 136.128, 'end': 141.68, 'duration': 5.552, 'lang': 'zh', 'lid_conf': 0.928, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 142.979, 'end': 171.751, 'duration': 28.772, 'lang': 'zh', 'lid_conf': 0.998, 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 171.751, 'end': 198.953, 'duration': 27.203, 'lang

'./out/speech_lang_timeline.csv'

## 6. 整理轉寫、對齊與輸出結果

這幾個 cell 會把每個切片的批次處理結果攤平成單一清單，輸出成全域時間的 JSON、CSV 與 SRT。

輸出內容包含：
- `chunk_checkpoints/*.json`：每個 5 分鐘切片各自一份 checkpoint，可用來斷點續跑。
- `asr_blocks.json`：實際拿去轉寫的區段。
- `bilingual_transcript.csv`：逐段轉寫結果，其中中文會依停頓合併成較長段落；藏語列的 `text_zh` 會是直接根據藏語發音產生的漢語直譯，欄位包含 `text_bo`、`text_zh` 與 `source`，且簡體中文會先轉成繁體。
- `bilingual_transcript.json`：保留同樣的段落化結構，方便後續程式處理；中文內容同樣會先轉成繁體。
- `bilingual_transcript_unitcode.json`：把 ASCII / Wylie 形式的 `text_bo` 轉成 Unicode 的 JSON。
- `bilingual_transcript.srt`：會把藏文與中文各自整理成較短小段落，且只在同語言內合併；藏語段落仍保留 `[藏文]` + `[中譯]` 的藏中對照。
- `bilingual_transcript.rtf`：每個時間段一個小段落；藏語段內保留多組 `藏文：... / 漢譯：...` 對照，中文段則獨立成段，不和藏語混排。

所有 `start` / `end` 都已經換算回原始長音檔的時間軸；如果本地 alignment 模型載入失敗，流程仍然會繼續，只是會退回原始 ASR 的時間戳。


In [12]:
asr_blocks = []
for result in chunk_results:
    asr_blocks.extend(result["asr_blocks"])

asr_blocks.sort(key=lambda x: (x["start"], x["end"]))
print("ASR blocks:", len(asr_blocks))
asr_blocks[:5]


ASR blocks: 140


[{'start': 1.465,
  'end': 29.444,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 30.524,
  'end': 56.663,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 57.355,
  'end': 84.068,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 84.44,
  'end': 105.719,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'},
 {'start': 107.035,
  'end': 135.689,
  'lang': 'bo',
  'chunk_index': 0,
  'audio_file': '0_0_300.mp3'}]

In [13]:
asr_blocks_json_path = os.path.join(OUT_DIR, "asr_blocks.json")
with open(asr_blocks_json_path, "w", encoding="utf-8") as f:
    json.dump(asr_blocks, f, ensure_ascii=False, indent=2)

asr_blocks_json_path


'./out/asr_blocks.json'

In [14]:
transcript_rows_raw = []
for result in chunk_results:
    transcript_rows_raw.extend(result["transcript_rows"])

if not TRANSCRIBE_ZH_INTERPRETER:
    transcript_rows_raw = [r for r in transcript_rows_raw if r.get("lang") != "zh"]

transcript_rows_raw.sort(key=lambda x: (x["start"], x["end"]))
transcript_rows_raw = convert_rows_zh_to_traditional(transcript_rows_raw, ZH_TRAD_CONVERTER)
transcript_rows = merge_transcript_rows(transcript_rows_raw, zh_max_gap=1.2, zh_max_block_sec=90.0)
print("transcript rows (raw):", len(transcript_rows_raw))
print("transcript rows (paragraph):", len(transcript_rows))
print(transcript_rows[:10])

# 2) 逐字稿 CSV + JSON
transcript_csv = os.path.join(OUT_DIR, "bilingual_transcript.csv")
write_csv(
    transcript_rows,
    transcript_csv,
    ["chunk_index", "audio_file", "start", "end", "lang", "source", "text_bo", "text_zh", "text"],
)

transcript_json = os.path.join(OUT_DIR, "bilingual_transcript.json")
with open(transcript_json, "w", encoding="utf-8") as f:
    json.dump(transcript_rows, f, ensure_ascii=False, indent=2)

transcript_unicode_rows = build_unicode_transcript_rows(transcript_rows, EWTS_CONVERTER)
transcript_unicode_json = os.path.join(OUT_DIR, "bilingual_transcript_unitcode.json")
with open(transcript_unicode_json, "w", encoding="utf-8") as f:
    json.dump(transcript_unicode_rows, f, ensure_ascii=False, indent=2)

transcript rows (raw): 1388
transcript rows (paragraph): 341
[{'start': 0.0, 'end': 5.0, 'lang': 'bo', 'source': 'vertexai_mixed', 'text': '在聽法之前，講一下動機和行為的次第。', 'text_bo': 'ཉན་ལེ་ལ་མ་བཞུགས་གོང་ལ་ཨ་ནི་ཀུན་སློང་ཀུན་སྤྱོད་ཀྱི་རིམ་པ་ཅིག་བཤད།', 'text_zh': '在聽法之前，講一下動機和行為的次第。', 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 5.0, 'end': 10.0, 'lang': 'bo', 'source': 'vertexai_mixed', 'text': '動機，我平時一直在講。就是好心，對吧。', 'text_bo': 'ད་ཀུན་སློང་ད་རྒྱུན་ལྡན་ནས་ངས་བཤད་བསྡད་ཀྱི་ཡོད་རེད། ཨ་ནི་བསམ་པ་བཟང་པོ་རེད་བ།', 'text_zh': '動機，我平時一直在講。就是好心，對吧。', 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 10.0, 'end': 15.0, 'lang': 'bo', 'source': 'vertexai_mixed', 'text': '對他人，不死就要利益的，不殺人的，生起這樣的心。', 'text_bo': 'གཞན་དེ་ལ་མ་ཤི་ཕན་དགོས་བསམ་པའི་འདྲ་པོ་མི་གསོད་ཡ་དེ་འདྲའི་བློ་ཅིག་ཐོག', 'text_zh': '對他人，不死就要利益的，不殺人的，生起這樣的心。', 'chunk_index': 0, 'audio_file': '0_0_300.mp3'}, {'start': 15.0, 'end': 20.0, 'lang': 'bo', 'source': 'vertexai_mixed', 'text': '生起這樣的心。然後遇到了大乘法。', 'text_bo': 'དེ་འདྲའི་བློ་ཅིག་ཐོ

In [15]:
# 3) 合併後 SRT + RTF
srt_rows = merge_subtitle_rows(
    transcript_rows_raw,
    zh_max_gap=0.5,
    zh_max_block_sec=12.0,
    bo_max_gap=0.8,
    bo_max_block_sec=12.0,
)
srt_path = os.path.join(OUT_DIR, "bilingual_transcript.srt")
write_srt(srt_rows, srt_path)

rtf_source_rows = build_unicode_transcript_rows(transcript_rows_raw, EWTS_CONVERTER)
rtf_sections = build_rtf_sections(
    rtf_source_rows,
    zh_max_gap=1.2,
    zh_max_block_sec=90.0,
    bo_max_gap=1.2,
    bo_max_block_sec=90.0,
)
rtf_path = os.path.join(OUT_DIR, "bilingual_transcript.rtf")
write_rtf_sections(rtf_sections, rtf_path)

print("Done")
print(" -", timeline_csv)
print(" -", asr_blocks_json_path)
print(" -", transcript_csv)
print(" -", transcript_json)
print(" -", transcript_unicode_json)
print(" -", srt_path)
print(" -", rtf_path)


Done
 - ./out/speech_lang_timeline.csv
 - ./out/asr_blocks.json
 - ./out/bilingual_transcript.csv
 - ./out/bilingual_transcript.json
 - ./out/bilingual_transcript_unitcode.json
 - ./out/bilingual_transcript.srt
 - ./out/bilingual_transcript.rtf
